In [2]:
!pip install aiforge-framework


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.115.6 requires starlette<0.42.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
gradio 6.5.1 requires starlette<1.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
langchain-community 0.3.14 requires langchain<0.4.0,>=0.3.14, but you have langchain 1.4.2 which is incompatible.
langchain-community 0.3.14 requires langchain-core<0.4.0,>=0.3.29, but you have langchain-core 1.6.4 which is incompatible.
langchain-community 0.3.14 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.14.0 which is incompatible.
langchain-text-splitters 0.3.4 requires langchain-core<0.4.0,>=0.3.26, but you have langchain-core 1.6.4 which is incompatible.
roboflow 1.2.13 requires idna==3.7, but you have idna 3.20 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1


  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached ormsgpack-1.12.2-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached pywin32-312-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached starlette-1.6.0-py3-none-any.whl.metadata (6.4 kB)
   ---------------------------------------- 0.0/572.0 kB ? eta -:--:--
   ---------------------------------------- 572.0/572.0 kB 9.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 17.9 MB/s  0:00:00
Using cached langchain_ollama-1.1.0-py3-none-any.whl (31 kB)

# utils

# Quiz Generation

In [3]:
from aipipe import LLM

In [4]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    r"C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe"
)

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())

CONFIG_DIR = PROJECT_ROOT / "config"
LLM_CONFIG = CONFIG_DIR / "llm.yaml"

print("LLM config:", LLM_CONFIG)
print("LLM config exists:", LLM_CONFIG.exists())

Project root: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe
Working directory: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe
LLM config: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe\config\llm.yaml
LLM config exists: True


In [5]:
from __future__ import annotations

import json
import os
from io import BytesIO
from pathlib import Path
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

load_dotenv(PROJECT_ROOT / ".env")

True

In [6]:
from aipipe import LLM
from aipipe.exceptions import SecurityError
from aipipe.security import SecurityManager

In [7]:
from quiz_ai import QuizAgent, QuizGenerator
from quiz_ai.security import (
    QuizSecurityPolicy,
    SecurityAgent,
)

In [8]:
Difficulty = Literal[
    "easy",
    "medium",
    "hard",
]

BloomLevel = Literal[
    "remember",
    "understand",
    "apply",
    "analyze",
    "evaluate",
    "create",
]

VALID_DIFFICULTIES = {
    "easy",
    "medium",
    "hard",
}

VALID_BLOOM_LEVELS = {
    "remember",
    "understand",
    "apply",
    "analyze",
    "evaluate",
    "create",
}

DEFAULT_COUNT = 10

DEFAULT_DIFFICULTY = {
    "easy": 3,
    "medium": 4,
    "hard": 3,
}

DEFAULT_BLOOM = {
    "remember": 2,
    "understand": 2,
    "apply": 2,
    "analyze": 2,
    "evaluate": 1,
    "create": 1,
}

In [9]:
class QuizGenerationRequest(BaseModel):

    content: str = Field(
        ...,
        min_length=1,
        description="Learning content.",
    )

    count: int = Field(
        default=5,
        ge=1,
        le=100,
        description="Number of questions.",
    )

    difficulty: dict[Difficulty, int] | None = None

    bloom: dict[BloomLevel, int] | None = None

    @field_validator("content")
    @classmethod
    def validate_content(cls, value: str) -> str:

        value = value.strip()

        if not value:
            raise ValueError("content cannot be empty.")

        return value

    @field_validator("difficulty")
    @classmethod
    def validate_difficulty(
        cls,
        value: dict[Difficulty, int] | None,
    ):
        if value is None:
            return None

        for key, amount in value.items():

            if not isinstance(amount, int):
                raise ValueError(
                    f"Difficulty '{key}' must be an integer."
                )

            if amount < 0:
                raise ValueError(
                    f"Difficulty '{key}' cannot be negative."
                )

        return value

    @field_validator("bloom")
    @classmethod
    def validate_bloom(
        cls,
        value: dict[BloomLevel, int] | None,
    ):
        if value is None:
            return None

        for key, amount in value.items():

            if not isinstance(amount, int):
                raise ValueError(
                    f"Bloom level '{key}' must be an integer."
                )

            if amount < 0:
                raise ValueError(
                    f"Bloom level '{key}' cannot be negative."
                )

        return value

In [10]:
def resolve_distributions(
    count: int,
    difficulty: dict[str, int] | None = None,
    bloom: dict[str, int] | None = None,
):
    if difficulty is None:

        if count == DEFAULT_COUNT:
            difficulty = DEFAULT_DIFFICULTY.copy()
        else:
            difficulty = {
                "medium": count
            }

    if bloom is None:

        if count == DEFAULT_COUNT:
            bloom = DEFAULT_BLOOM.copy()
        else:
            bloom = {
                "understand": count
            }

    return difficulty, bloom

In [11]:
def validate_generation_request(
    content: str,
    count: int,
    difficulty: dict[str, int],
    bloom: dict[str, int],
) -> None:

    if not isinstance(content, str):
        raise TypeError("content must be a string.")

    content = content.strip()

    if not content:
        raise ValueError("content cannot be empty.")

    if count < 1:
        raise ValueError("count must be greater than 0.")

    if count > 100:
        raise ValueError(
            "count cannot be greater than 100."
        )

    unknown_difficulty = (
        set(difficulty) - VALID_DIFFICULTIES
    )

    if unknown_difficulty:
        raise ValueError(
            f"Invalid difficulty values: "
            f"{sorted(unknown_difficulty)}"
        )

    unknown_bloom = (
        set(bloom) - VALID_BLOOM_LEVELS
    )

    if unknown_bloom:
        raise ValueError(
            f"Invalid Bloom levels: "
            f"{sorted(unknown_bloom)}"
        )

    if any(
        not isinstance(value, int)
        or value < 0
        for value in difficulty.values()
    ):
        raise ValueError(
            "Difficulty counts must be non-negative integers."
        )

    if any(
        not isinstance(value, int)
        or value < 0
        for value in bloom.values()
    ):
        raise ValueError(
            "Bloom counts must be non-negative integers."
        )

    if sum(difficulty.values()) != count:
        raise ValueError(
            "Difficulty distribution must equal question count. "
            f"Expected {count}, "
            f"got {sum(difficulty.values())}."
        )

    if sum(bloom.values()) != count:
        raise ValueError(
            "Bloom distribution must equal question count. "
            f"Expected {count}, "
            f"got {sum(bloom.values())}."
        )

In [12]:
def create_security_agent() -> SecurityAgent:

    security_manager = SecurityManager()

    policy = QuizSecurityPolicy(
        enabled=True,
        block_jailbreak=True,
        block_document_injection=True,
        block_quiz_manipulation=True,
        block_tool_abuse=True,
        block_role_override=True,
        block_system_prompt_extraction=True,
        block_secret_extraction=True,
        fail_closed=True,
    )

    return SecurityAgent(
        security_manager=security_manager,
        policy=policy,
    )

In [13]:
def generate_secure_quiz(
    content: str,
    count: int,
    difficulty: dict[str, int],
    bloom: dict[str, int],
) -> dict[str, Any]:

    # 1. Validate request
    validate_generation_request(
        content=content,
        count=count,
        difficulty=difficulty,
        bloom=bloom,
    )

    print("=" * 80)
    print("SECURE QUIZ GENERATION")
    print("=" * 80)

    print(f"Characters : {len(content)}")
    print(f"Words      : {len(content.split())}")
    print(f"Questions  : {count}")
    print(f"Difficulty : {difficulty}")
    print(f"Bloom      : {bloom}")

    # 2. Security
    print("\n1. SECURITY INITIALIZATION")

    security_agent = create_security_agent()

    print("SecurityAgent : READY")

    # 3. Check learning content
    print("\n2. LEARNING CONTENT SECURITY CHECK")

    security_result = (
        security_agent.check_learning_content(
            content
        )
    )

    print("Allowed    :", security_result.allowed)
    print("Risk       :", security_result.risk)
    print("Categories :", security_result.categories)
    print("Message    :", security_result.message)

    if not security_result.allowed:
        raise SecurityError(
            security_result.message
            or "Learning content blocked by security policy.",
            result=security_result,
        )

    # 4. Sanitize
    print("\n3. CONTENT SANITIZATION")

    safe_content = (
        security_agent.secure_learning_content(
            content
        )
    )

    print(
        "Sanitized characters :",
        len(safe_content),
    )

    # 5. Load LLM
    print("\n4. LOAD LLM")

    if not LLM_CONFIG.exists():
        raise FileNotFoundError(
            f"LLM configuration not found: {LLM_CONFIG}"
        )

    llm = LLM.from_config(
        str(LLM_CONFIG)
    )

    print("LLM:", llm.info())

    # 6. Quiz Agent
    print("\n5. CREATE QUIZ AGENT")

    agent = QuizAgent(
        llm=llm,
        tools=None,
        memory=None,
        security=security_agent,
        name="secure_quiz_agent",
    )

    print("QuizAgent : READY")

    # 7. Quiz Generator
    print("\n6. CREATE QUIZ GENERATOR")

    generator = QuizGenerator(
        agent=agent,
        batch_size=min(10, count),
        max_retries=2,
    )

    print("QuizGenerator : READY")

    # 8. Generate
    print("\n7. GENERATING QUIZ")

    try:

        generated_quiz = generator.generate(
            content=safe_content,
            count=count,
            difficulty=difficulty,
            bloom=bloom,
        )

    except SecurityError:
        raise

    except Exception as exc:
        raise RuntimeError(
            f"Quiz generation failed: {exc}"
        ) from exc

    print("LLM generation : COMPLETE")

    return generated_quiz

In [14]:
content = """
Python is a high-level programming language.
It is widely used for web development, data science,
machine learning, automation, and scripting.

Python uses variables to store data.
Common data types include strings, integers,
floating-point numbers, lists, tuples, dictionaries,
and sets.

Python supports conditional statements such as if,
elif, and else. It also supports loops such as
for and while.
"""

count = 5

difficulty = {
    "easy": 2,
    "medium": 2,
    "hard": 1,
}

bloom = {
    "remember": 1,
    "understand": 1,
    "apply": 1,
    "analyze": 1,
    "evaluate": 1,
    "create": 0,
}

result = generate_secure_quiz(
    content=content,
    count=count,
    difficulty=difficulty,
    bloom=bloom,
)

SECURE QUIZ GENERATION
Characters : 404
Words      : 58
Questions  : 5
Difficulty : {'easy': 2, 'medium': 2, 'hard': 1}
Bloom      : {'remember': 1, 'understand': 1, 'apply': 1, 'analyze': 1, 'evaluate': 1, 'create': 0}

1. SECURITY INITIALIZATION
SecurityAgent : READY

2. LEARNING CONTENT SECURITY CHECK
Allowed    : True
Risk       : none
Categories : []
Message    : Security check passed.

3. CONTENT SANITIZATION
Sanitized characters : 404

4. LOAD LLM
LLM: {'provider': 'ollama', 'model': 'gemma4:31b', 'api_key': None, 'api_key_env': 'OLLAMA_API_KEY', 'base_url': 'https://ollama.com', 'temperature': 0.0, 'max_tokens': 4096, 'generation_type': 'text', 'supports_streaming': True, 'supports_tools': True, 'supports_structured_output': True, 'timeout': 120.0, 'connection_mode': 'ollama_cloud', 'client': {'class': 'LLM', 'model_class': 'OllamaCloudAPIModel'}}

5. CREATE QUIZ AGENT
QuizAgent : READY

6. CREATE QUIZ GENERATOR
QuizGenerator : READY

7. GENERATING QUIZ
SECURE QUIZ BATCH GENERA

# personalized learning

Set project root


In [78]:
# ============================================================
# CELL 1 - Imports and Project Configuration
# ============================================================

import os
import sys
import json
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [79]:
# ============================================================
# CELL 2 - Project Paths
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)
print("Current Directory:", Path.cwd())

Project Root: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe
Current Directory: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe


In [80]:
# ============================================================
# CELL 3 - Database Configuration
# ============================================================

DATABASE_URL = (
    "mysql+pymysql://root:"
    "Mysql%402025"
    "@127.0.0.1:3306/student_ai_demo"
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True
)

print("✅ Database engine created")

✅ Database engine created


In [82]:
# ============================================================
# CELL 4 - Test Database
# ============================================================

def test_database_connection():
    try:
        with engine.connect() as connection:
            result = connection.execute(text("SELECT 1"))
            value = result.scalar()

        print("✅ MySQL connection successful!")
        print("Result:", value)

        return True

    except Exception as e:
        print("❌ Database connection failed")
        print(e)

        return False


db_connected = test_database_connection()

✅ MySQL connection successful!
Result: 1


In [83]:
# ============================================================
# CELL 5 - Get Student
# ============================================================

def get_student(student_id):
    query = text("""
        SELECT
            id,
            name,
            email,
            batch
        FROM students
        WHERE id = :student_id
    """)

    with engine.connect() as connection:
        result = connection.execute(
            query,
            {"student_id": student_id}
        ).mappings().first()

    if result is None:
        raise ValueError(
            f"Student with ID {student_id} not found"
        )

    return dict(result)

In [84]:
student = get_student(1)

print(student)

{'id': 1, 'name': 'Demo Student', 'email': 'student1@example.com', 'batch': 'Aarohan-2026'}


In [85]:
# ============================================================
# CELL 6 - Get Quiz Attempts
# ============================================================

def get_quiz_attempts(student_id):
    
    query = text("""
        SELECT
            qa.id AS attempt_id,
            qa.student_id,
            qa.quiz_id,
            qa.score,
            qa.total_questions,
            qa.correct_questions,
            qa.completed_at,

            q.title AS quiz_title,

            d.id AS day_id,
            d.day_number,

            m.id AS module_id,
            m.name AS module_name

        FROM quiz_attempts qa

        JOIN quizzes q
            ON qa.quiz_id = q.id

        JOIN days d
            ON q.day_id = d.id

        JOIN modules m
            ON d.module_id = m.id

        WHERE qa.student_id = :student_id

        ORDER BY qa.completed_at
    """)

    with engine.connect() as connection:
        result = connection.execute(
            query,
            {"student_id": student_id}
        )

        rows = result.mappings().all()

    return pd.DataFrame(rows)

In [86]:
quiz_attempts_df = get_quiz_attempts(1)

display(quiz_attempts_df)

,attempt_id,completed_at,correct_questions,day_id,day_number,module_id,module_name,quiz_id,quiz_title,score,student_id,total_questions
0,6,2026-09-01 10:00:00,2,6,1,1,Python Basics,6,Day 1 Quiz,40.00,1,5
1,5,2026-09-02 10:00:00,3,5,2,1,Python Basics,5,Day 2 Quiz,60.00,1,5
2,4,2026-09-03 10:00:00,3,4,3,1,Python Basics,4,Day 3 Quiz,55.00,1,5
3,3,2026-09-04 10:00:00,4,3,4,1,Python Basics,3,Day 4 Quiz,70.00,1,5
4,2,2026-09-05 10:00:00,4,2,5,1,Python Basics,2,Day 5 Quiz,75.00,1,5
5,1,2026-09-06 10:00:00,4,1,6,1,Python Basics,1,Day 6 Quiz,80.00,1,5


In [87]:
# ============================================================
# CELL 7 - Calculate Overall Performance
# ============================================================

def calculate_overall_score(quiz_attempts_df):

    if quiz_attempts_df.empty:
        return 0.0

    total_questions = quiz_attempts_df[
        "total_questions"
    ].sum()

    correct_questions = quiz_attempts_df[
        "correct_questions"
    ].sum()

    if total_questions == 0:
        return 0.0

    score = (
        correct_questions /
        total_questions
    ) * 100

    return round(score, 2)

In [88]:
overall_score = calculate_overall_score(
    quiz_attempts_df
)

print("Overall Score:", overall_score, "%")

Overall Score: 66.67 %


In [89]:
# ============================================================
# CELL 8 - Risk Level
# ============================================================

def calculate_risk_level(score):

    if score < 50:
        return "high"

    elif score < 70:
        return "medium"

    else:
        return "low"

In [90]:
risk_level = calculate_risk_level(
    overall_score
)

print("Risk Level:", risk_level)

Risk Level: medium


In [91]:
# ============================================================
# CELL 9 - Question-Level Analysis
# ============================================================

def get_question_analysis(student_id):

    query = text("""
        SELECT

            qa.id AS question_attempt_id,

            qa.attempt_id,

            qa.question_id,

            qa.student_answer,

            qa.is_correct,

            q.question_text,

            q.correct_answer,

            t.id AS topic_id,

            t.name AS topic,

            l.id AS lesson_id,

            l.title AS lesson_title,

            d.id AS day_id,

            d.day_number,

            m.id AS module_id,

            m.name AS module_name,

            quiz.title AS quiz_title

        FROM question_attempts qa

        JOIN questions q
            ON qa.question_id = q.id

        JOIN topics t
            ON q.topic_id = t.id

        JOIN lessons l
            ON t.lesson_id = l.id

        JOIN days d
            ON l.day_id = d.id

        JOIN modules m
            ON d.module_id = m.id

        JOIN quizzes quiz
            ON q.quiz_id = quiz.id

        JOIN quiz_attempts attempt
            ON qa.attempt_id = attempt.id

        WHERE attempt.student_id = :student_id

        ORDER BY
            attempt.completed_at,
            qa.question_id
    """)

    with engine.connect() as connection:
        result = connection.execute(
            query,
            {"student_id": student_id}
        )

        rows = result.mappings().all()

    return pd.DataFrame(rows)

In [92]:
question_df = get_question_analysis(1)

display(question_df)

,attempt_id,correct_answer,day_id,day_number,is_correct,lesson_id,lesson_title,module_id,module_name,question_attempt_id,question_id,question_text,quiz_title,student_answer,topic,topic_id
0,6,C,6,1,0,6,Lesson 1 - Core Concepts,1,Python Basics,26,26,Question 5 for Day 1 Quiz,Day 1 Quiz,Wrong,Variables,6
1,6,B,6,1,0,6,Lesson 1 - Core Concepts,1,Python Basics,27,27,Question 4 for Day 1 Quiz,Day 1 Quiz,A,Variables,6
2,6,A,6,1,0,6,Lesson 1 - Core Concepts,1,Python Basics,28,28,Question 3 for Day 1 Quiz,Day 1 Quiz,A,Variables,6
3,6,C,6,1,1,6,Lesson 1 - Core Concepts,1,Python Basics,29,29,Question 2 for Day 1 Quiz,Day 1 Quiz,C,Variables,6
4,6,B,6,1,1,6,Lesson 1 - Core Concepts,1,Python Basics,30,30,Question 1 for Day 1 Quiz,Day 1 Quiz,B,Variables,6
5,5,C,5,2,1,5,Lesson 2 - Core Concepts,1,Python Basics,21,21,Question 5 for Day 2 Quiz,Day 2 Quiz,C,Functions,5
6,5,B,5,2,1,5,Lesson 2 - Core Concepts,1,Python Basics,22,22,Question 4 for Day 2 Quiz,Day 2 Quiz,B,Functions,5
7,5,A,5,2,1,5,Lesson 2 - Core Concepts,1,Python Basics,23,23,Question 3 for Day 2 Quiz,Day 2 Quiz,A,Functions,5
8,5,C,5,2,1,5,Lesson 2 - Core Concepts,1,Python Basics,24,24,Question 2 for Day 2 Quiz,Day 2 Quiz,C,Functions,5
9,5,B,5,2,1,5,Lesson 2 - Core Concepts,1,Python Basics,25,25,Question 1 for Day 2 Quiz,Day 2 Quiz,B,Functions,5


In [93]:
# ============================================================
# CELL 10 - Get Wrong Questions
# ============================================================

def get_wrong_questions(question_df):

    if question_df.empty:
        return pd.DataFrame()

    wrong_questions = question_df[
        question_df["is_correct"] == 0
    ].copy()

    return wrong_questions

In [94]:
wrong_questions_df = get_wrong_questions(
    question_df
)

print(
    "Number of wrong questions:",
    len(wrong_questions_df)
)

display(wrong_questions_df)

Number of wrong questions: 4


,attempt_id,correct_answer,day_id,day_number,is_correct,lesson_id,lesson_title,module_id,module_name,question_attempt_id,question_id,question_text,quiz_title,student_answer,topic,topic_id
0,6,C,6,1,0,6,Lesson 1 - Core Concepts,1,Python Basics,26,26,Question 5 for Day 1 Quiz,Day 1 Quiz,Wrong,Variables,6
1,6,B,6,1,0,6,Lesson 1 - Core Concepts,1,Python Basics,27,27,Question 4 for Day 1 Quiz,Day 1 Quiz,A,Variables,6
2,6,A,6,1,0,6,Lesson 1 - Core Concepts,1,Python Basics,28,28,Question 3 for Day 1 Quiz,Day 1 Quiz,A,Variables,6
13,4,C,4,3,0,4,Lesson 3 - Core Concepts,1,Python Basics,19,19,Question 2 for Day 3 Quiz,Day 3 Quiz,Wrong,Data Types,4


In [95]:
# ============================================================
# CELL 11 - Weak Topic Analysis
# ============================================================

def calculate_topic_performance(question_df):

    if question_df.empty:
        return pd.DataFrame()

    topic_performance = (
        question_df
        .groupby(
            ["topic_id", "topic"],
            as_index=False
        )
        .agg(
            total_questions=(
                "question_id",
                "count"
            ),
            correct_questions=(
                "is_correct",
                "sum"
            )
        )
    )

    topic_performance["accuracy"] = (
        topic_performance["correct_questions"]
        /
        topic_performance["total_questions"]
        * 100
    )

    topic_performance["accuracy"] = (
        topic_performance["accuracy"]
        .round(2)
    )

    return topic_performance

In [96]:
topic_performance_df = calculate_topic_performance(
    question_df
)

display(topic_performance_df)

,topic_id,topic,total_questions,correct_questions,accuracy
0,1,Data Types,5,5,100.0
1,2,Functions,5,5,100.0
2,3,Variables,5,5,100.0
3,4,Data Types,5,4,80.0
4,5,Functions,5,5,100.0
5,6,Variables,5,2,40.0


In [97]:
# ============================================================
# CELL 12 - Identify Weak Topics
# ============================================================

def identify_weak_topics(
    topic_performance_df,
    threshold=70
):

    if topic_performance_df.empty:
        return pd.DataFrame()

    weak_topics = topic_performance_df[
        topic_performance_df["accuracy"] < threshold
    ].copy()

    weak_topics = weak_topics.sort_values(
        by="accuracy",
        ascending=True
    )

    return weak_topics

In [98]:
weak_topics_df = identify_weak_topics(
    topic_performance_df
)

print("Weak Topics:")

display(weak_topics_df)

Weak Topics:


,topic_id,topic,total_questions,correct_questions,accuracy
5,6,Variables,5,2,40.0


In [99]:
# ============================================================
# CELL 13 - Prepare AI Input
# ============================================================

def prepare_wrong_questions_for_ai(
    wrong_questions_df
):

    wrong_question_list = []

    for _, row in wrong_questions_df.iterrows():

        wrong_question_list.append({
            "question_id": int(
                row["question_id"]
            ),

            "question": str(
                row["question_text"]
            ),

            "student_answer": str(
                row["student_answer"]
            ),

            "correct_answer": str(
                row["correct_answer"]
            ),

            "topic": str(
                row["topic"]
            )
        })

    return wrong_question_list

In [101]:
# ============================================================
# CELL 14 - Primary Weak Topic
# ============================================================

def get_primary_weak_topic(
    weak_topics_df
):

    if weak_topics_df.empty:
        return "General Concepts"

    return str(
        weak_topics_df.iloc[0]["topic"]
    )

In [102]:
primary_topic = get_primary_weak_topic(
    weak_topics_df
)

print("Primary Weak Topic:", primary_topic)

Primary Weak Topic: Variables


In [103]:
# ============================================================
# CELL 15 - Load Environment
# ============================================================

load_dotenv(
    PROJECT_ROOT / ".env"
)

print(
    "OLLAMA_API_KEY available:",
    bool(os.getenv("OLLAMA_API_KEY"))
)

OLLAMA_API_KEY available: True


In [104]:
# ============================================================
# CELL 16 - Load AIPipe LLM
# ============================================================

from aipipe import LLM

config_file = (
    PROJECT_ROOT /
    "config" /
    "llm.yaml"
)

if not config_file.exists():
    raise FileNotFoundError(
        f"LLM config not found: {config_file}"
    )

llm = LLM.from_config(
    config_file
)

print("✅ AIPipe LLM initialized")

✅ AIPipe LLM initialized


In [105]:
# ============================================================
# CELL 17 - Build Personalized Prompt
# ============================================================

def build_personalized_prompt(
    student,
    overall_score,
    risk_level,
    primary_topic,
    wrong_question_list
):

    wrong_questions_json = json.dumps(
        wrong_question_list,
        indent=2,
        default=str
    )

    prompt = f"""
You are a Personalized AI Learning Assistant
inside an LMS.

Analyze the student's quiz performance and create
a personalized learning recommendation.

STUDENT
-------
Name: {student["name"]}
Overall Score: {overall_score:.2f}%
Risk Level: {risk_level}

WEAK TOPIC
----------
{primary_topic}

WRONG QUESTIONS
---------------
{wrong_questions_json}

TASK
----
1. Identify the main learning gap.
2. Explain the weak topic in very simple language.
3. Explain it like teaching a 5-year-old.
4. Explain the likely misunderstanding.
5. Give a real-world analogy.
6. Give a simple coding example if applicable.
7. Recommend the lesson/topic to study.
8. Give one practical activity.
9. Generate 3 practice questions.
10. Give a short motivational message.

IMPORTANT
---------
Focus only on the student's weak topic.
Do not recommend unrelated topics.

Return ONLY valid JSON.

FORMAT
------
{{
    "student": "",
    "risk_level": "",
    "weak_topic": "",
    "learning_gap": "",
    "simple_explanation": "",
    "five_year_old_explanation": "",
    "real_world_analogy": "",
    "coding_example": "",
    "recommended_lesson": "",
    "practice_activity": "",
    "practice_questions": [],
    "motivation": ""
}}
"""

    return prompt

In [107]:
# ============================================================
# CELL 18 - Generate Personalized AI Response
# ============================================================

def generate_personalized_response(
    llm,
    prompt
):

    response = llm.generate(prompt)

    return response

In [108]:
prompt = build_personalized_prompt(
    student=student,
    overall_score=overall_score,
    risk_level=risk_level,
    primary_topic=primary_topic,
    wrong_question_list=wrong_question_list
)

ai_response = generate_personalized_response(
    llm,
    prompt
)

print(ai_response)

```json
{
    "student": "Demo Student",
    "risk_level": "medium",
    "weak_topic": "Variables",
    "learning_gap": "The student is struggling with the fundamental concept of variables, specifically how they are declared, assigned, or updated, as evidenced by multiple incorrect answers in the Day 1 Quiz.",
    "simple_explanation": "A variable is like a labeled container that holds a piece of information. Instead of remembering the information itself, you just remember the name of the container to get the value inside.",
    "five_year_old_explanation": "Imagine you have a bunch of toy boxes. You put a toy car in one box and write 'My Toy' on the outside. Whenever you want your car, you just look for the box that says 'My Toy'!",
    "real_world_analogy": "Think of a variable like a contact name in your phone. The name 'Mom' is the variable, and the actual phone number is the value stored inside. If Mom changes her number, you just update the value inside the 'Mom' contact, but you

In [109]:
# ============================================================
# CELL 19 - Extract AI Text
# ============================================================

def extract_ai_text(ai_response):

    if hasattr(ai_response, "content"):
        return ai_response.content

    elif hasattr(ai_response, "text"):
        return ai_response.text

    elif hasattr(ai_response, "response"):
        return ai_response.response

    return str(ai_response)

In [110]:
ai_text = extract_ai_text(
    ai_response
)

print(ai_text)

```json
{
    "student": "Demo Student",
    "risk_level": "medium",
    "weak_topic": "Variables",
    "learning_gap": "The student is struggling with the fundamental concept of variables, specifically how they are declared, assigned, or updated, as evidenced by multiple incorrect answers in the Day 1 Quiz.",
    "simple_explanation": "A variable is like a labeled container that holds a piece of information. Instead of remembering the information itself, you just remember the name of the container to get the value inside.",
    "five_year_old_explanation": "Imagine you have a bunch of toy boxes. You put a toy car in one box and write 'My Toy' on the outside. Whenever you want your car, you just look for the box that says 'My Toy'!",
    "real_world_analogy": "Think of a variable like a contact name in your phone. The name 'Mom' is the variable, and the actual phone number is the value stored inside. If Mom changes her number, you just update the value inside the 'Mom' contact, but you

In [111]:
# ============================================================
# CELL 20 - Parse AI JSON
# ============================================================

def parse_ai_json(ai_text):

    cleaned_text = ai_text.strip()

    # Remove Markdown JSON fences
    cleaned_text = re.sub(
        r"^```json\s*",
        "",
        cleaned_text,
        flags=re.IGNORECASE
    )

    cleaned_text = re.sub(
        r"\s*```$",
        "",
        cleaned_text
    )

    cleaned_text = cleaned_text.strip()

    try:

        result = json.loads(
            cleaned_text
        )

        return result

    except json.JSONDecodeError as e:

        print("❌ JSON parsing failed")
        print("Error:", e)

        print("\nAI Response:")
        print(cleaned_text)

        return None

In [112]:
personalized_result = parse_ai_json(
    ai_text
)

if personalized_result:

    print(
        "✅ Personalized AI result created"
    )

✅ Personalized AI result created


In [1]:
# ============================================================
# CELL 21 - Display Personalized Result
# ============================================================

def display_personalized_result(result):

    if not result:
        print("❌ No personalized result available")
        return

    print("=" * 60)
    print("PERSONALIZED LEARNING RECOMMENDATION")
    print("=" * 60)

    print("\n🎯 Weak Topic:")
    print(result.get("weak_topic"))

    print("\n📌 Learning Gap:")
    print(result.get("learning_gap"))

    print("\n📖 Simple Explanation:")
    print(result.get("simple_explanation"))

    print("\n👶 5-Year-Old Explanation:")
    print(result.get(
        "five_year_old_explanation"
    ))

    print("\n🌍 Real-World Analogy:")
    print(result.get(
        "real_world_analogy"
    ))

    print("\n💻 Coding Example:")
    print(result.get(
        "coding_example"
    ))

    print("\n📚 Recommended Lesson:")
    print(result.get(
        "recommended_lesson"
    ))

    print("\n🧪 Practice Activity:")
    print(result.get(
        "practice_activity"
    ))

    print("\n📝 Practice Questions:")

    for i, question in enumerate(
        result.get(
            "practice_questions",
            []
        ),
        1
    ):
        print(f"{i}. {question}")

    print("\n💪 Motivation:")
    print(result.get("motivation"))

In [5]:
# ============================================================
# PERSONALIZED AI - MAIN FUNCTION
# ============================================================

def run_personalized_ai(
    student_id,
    weak_topic_threshold=70
):
    """
    Complete Personalized AI pipeline.

    Flow:
    Student
        -> Quiz Attempts
        -> Overall Score
        -> Risk Level
        -> Question Analysis
        -> Wrong Questions
        -> Topic Performance
        -> Weak Topic
        -> Ollama AI
        -> Personalized Recommendation
    """

    print("=" * 60)
    print("STARTING PERSONALIZED AI ANALYSIS")
    print("=" * 60)

    # --------------------------------------------------------
    # 1. Get student
    # --------------------------------------------------------

    student = get_student(student_id)

    print(
        f"\n👤 Student: {student['name']}"
    )

    # --------------------------------------------------------
    # 2. Get quiz attempts
    # --------------------------------------------------------

    quiz_attempts = get_quiz_attempts(student_id)

    if quiz_attempts.empty:
        print("❌ No quiz attempts found.")
        return None

    print(
        f"📊 Quiz Attempts: {len(quiz_attempts)}"
    )

    # --------------------------------------------------------
    # 3. Calculate overall score
    # --------------------------------------------------------

    overall_score = calculate_overall_score(
        quiz_attempts
    )

    print(
        f"📈 Overall Score: {overall_score:.2f}%"
    )

    # --------------------------------------------------------
    # 4. Calculate risk
    # --------------------------------------------------------

    risk_level = calculate_risk_level(
        overall_score
    )

    print(
        f"⚠️ Risk Level: {risk_level}"
    )

    # --------------------------------------------------------
    # 5. Question-level analysis
    # --------------------------------------------------------

    question_df = get_question_analysis(
        student_id
    )

    if question_df.empty:
        print("❌ No question-level data found.")
        return None

    print(
        f"📝 Questions Analyzed: {len(question_df)}"
    )

    # --------------------------------------------------------
    # 6. Wrong questions
    # --------------------------------------------------------

    wrong_questions_df = get_wrong_questions(
        question_df
    )

    print(
        f"❌ Wrong Questions: {len(wrong_questions_df)}"
    )

    # --------------------------------------------------------
    # 7. Topic performance
    # --------------------------------------------------------

    topic_performance_df = (
        calculate_topic_performance(
            question_df
        )
    )

    # --------------------------------------------------------
    # 8. Identify weak topics
    # --------------------------------------------------------

    weak_topics_df = identify_weak_topics(
        topic_performance_df,
        threshold=weak_topic_threshold
    )

    print(
        f"🎯 Weak Topics Found: {len(weak_topics_df)}"
    )

    # --------------------------------------------------------
    # 9. Primary weak topic
    # --------------------------------------------------------

    primary_topic = get_primary_weak_topic(
        weak_topics_df
    )

    print(
        f"📚 Primary Weak Topic: {primary_topic}"
    )

    # --------------------------------------------------------
    # 10. Prepare wrong questions for AI
    # --------------------------------------------------------

    wrong_question_list = (
        prepare_wrong_questions_for_ai(
            wrong_questions_df
        )
    )

    # --------------------------------------------------------
    # 11. Build AI prompt
    # --------------------------------------------------------

    prompt = build_personalized_prompt(
        student=student,
        overall_score=overall_score,
        risk_level=risk_level,
        primary_topic=primary_topic,
        wrong_question_list=wrong_question_list
    )

    # --------------------------------------------------------
    # 12. Generate AI response
    # --------------------------------------------------------

    print("\n🤖 Generating personalized recommendation...")

    ai_response = generate_personalized_response(
        llm,
        prompt
    )

    # --------------------------------------------------------
    # 13. Extract text
    # --------------------------------------------------------

    ai_text = extract_ai_text(
        ai_response
    )

    # --------------------------------------------------------
    # 14. Parse JSON
    # --------------------------------------------------------

    personalized_result = parse_ai_json(
        ai_text
    )

    if personalized_result is None:

        print(
            "❌ Could not create personalized result."
        )

        return {
            "student": student,
            "overall_score": overall_score,
            "risk_level": risk_level,
            "quiz_attempts": quiz_attempts,
            "question_analysis": question_df,
            "wrong_questions": wrong_questions_df,
            "topic_performance": topic_performance_df,
            "weak_topics": weak_topics_df,
            "primary_topic": primary_topic,
            "ai_result": None
        }

    print(
        "\n✅ Personalized AI analysis completed!"
    )

    # --------------------------------------------------------
    # 15. Return complete result
    # --------------------------------------------------------

    return {
        "student": student,
        "overall_score": overall_score,
        "risk_level": risk_level,
        "quiz_attempts": quiz_attempts,
        "question_analysis": question_df,
        "wrong_questions": wrong_questions_df,
        "topic_performance": topic_performance_df,
        "weak_topics": weak_topics_df,
        "primary_topic": primary_topic,
        "ai_result": personalized_result
    }

In [6]:
print("run_personalized_ai" in globals())

if "run_personalized_ai" in globals():
    print("✅ Main function loaded successfully")
else:
    print("❌ Main function is not loaded")

True
✅ Main function loaded successfully


In [8]:
# ============================================================
# CELL - ALL PERSONALIZED AI HELPER FUNCTIONS
# ============================================================

import os
import sys
import json
import re
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv


# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 2. DATABASE CONNECTION
# ============================================================

DATABASE_URL = (
    "mysql+pymysql://root:"
    "Mysql%402025"
    "@127.0.0.1:3306/student_ai_demo"
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True
)


# ============================================================
# 3. GET STUDENT
# ============================================================

def get_student(student_id):

    query = text("""
        SELECT
            id,
            name,
            email,
            batch
        FROM students
        WHERE id = :student_id
    """)

    with engine.connect() as connection:

        result = connection.execute(
            query,
            {"student_id": student_id}
        ).mappings().first()

    if result is None:

        raise ValueError(
            f"Student with ID {student_id} not found"
        )

    return dict(result)


# ============================================================
# 4. GET QUIZ ATTEMPTS
# ============================================================

def get_quiz_attempts(student_id):

    query = text("""
        SELECT
            qa.id AS attempt_id,
            qa.student_id,
            qa.quiz_id,
            qa.score,
            qa.total_questions,
            qa.correct_questions,
            qa.completed_at,

            q.title AS quiz_title,

            d.id AS day_id,
            d.day_number,

            m.id AS module_id,
            m.name AS module_name

        FROM quiz_attempts qa

        JOIN quizzes q
            ON qa.quiz_id = q.id

        JOIN days d
            ON q.day_id = d.id

        JOIN modules m
            ON d.module_id = m.id

        WHERE qa.student_id = :student_id

        ORDER BY qa.completed_at
    """)

    with engine.connect() as connection:

        result = connection.execute(
            query,
            {"student_id": student_id}
        ).mappings().all()

    return pd.DataFrame(result)


# ============================================================
# 5. CALCULATE OVERALL SCORE
# ============================================================

def calculate_overall_score(
    quiz_attempts_df
):

    if quiz_attempts_df.empty:
        return 0.0

    total_questions = (
        quiz_attempts_df[
            "total_questions"
        ].sum()
    )

    correct_questions = (
        quiz_attempts_df[
            "correct_questions"
        ].sum()
    )

    if total_questions == 0:
        return 0.0

    score = (
        correct_questions /
        total_questions
    ) * 100

    return round(score, 2)


# ============================================================
# 6. CALCULATE RISK
# ============================================================

def calculate_risk_level(score):

    if score < 50:
        return "high"

    elif score < 70:
        return "medium"

    return "low"


# ============================================================
# 7. QUESTION LEVEL ANALYSIS
# ============================================================

def get_question_analysis(student_id):

    query = text("""
        SELECT

            qa.id AS question_attempt_id,

            qa.attempt_id,

            qa.question_id,

            qa.student_answer,

            qa.is_correct,

            q.question_text,

            q.correct_answer,

            t.id AS topic_id,

            t.name AS topic,

            l.id AS lesson_id,

            l.title AS lesson_title,

            d.id AS day_id,

            d.day_number,

            m.id AS module_id,

            m.name AS module_name,

            quiz.title AS quiz_title

        FROM question_attempts qa

        JOIN questions q
            ON qa.question_id = q.id

        JOIN topics t
            ON q.topic_id = t.id

        JOIN lessons l
            ON t.lesson_id = l.id

        JOIN days d
            ON l.day_id = d.id

        JOIN modules m
            ON d.module_id = m.id

        JOIN quizzes quiz
            ON q.quiz_id = quiz.id

        JOIN quiz_attempts attempt
            ON qa.attempt_id = attempt.id

        WHERE attempt.student_id = :student_id

        ORDER BY
            attempt.completed_at,
            qa.question_id
    """)

    with engine.connect() as connection:

        result = connection.execute(
            query,
            {"student_id": student_id}
        ).mappings().all()

    return pd.DataFrame(result)


# ============================================================
# 8. GET WRONG QUESTIONS
# ============================================================

def get_wrong_questions(
    question_df
):

    if question_df.empty:
        return pd.DataFrame()

    return question_df[
        question_df["is_correct"] == 0
    ].copy()


# ============================================================
# 9. TOPIC PERFORMANCE
# ============================================================

def calculate_topic_performance(
    question_df
):

    if question_df.empty:
        return pd.DataFrame()

    topic_performance = (
        question_df
        .groupby(
            ["topic_id", "topic"],
            as_index=False
        )
        .agg(
            total_questions=(
                "question_id",
                "count"
            ),

            correct_questions=(
                "is_correct",
                "sum"
            )
        )
    )

    topic_performance["accuracy"] = (
        topic_performance["correct_questions"]
        /
        topic_performance["total_questions"]
        * 100
    )

    topic_performance["accuracy"] = (
        topic_performance["accuracy"]
        .round(2)
    )

    return topic_performance


# ============================================================
# 10. IDENTIFY WEAK TOPICS
# ============================================================

def identify_weak_topics(
    topic_performance_df,
    threshold=70
):

    if topic_performance_df.empty:
        return pd.DataFrame()

    weak_topics = topic_performance_df[
        topic_performance_df["accuracy"] < threshold
    ].copy()

    return weak_topics.sort_values(
        by="accuracy",
        ascending=True
    )


# ============================================================
# 11. PREPARE WRONG QUESTIONS FOR AI
# ============================================================

def prepare_wrong_questions_for_ai(
    wrong_questions_df
):

    wrong_question_list = []

    for _, row in wrong_questions_df.iterrows():

        wrong_question_list.append({

            "question_id": int(
                row["question_id"]
            ),

            "question": str(
                row["question_text"]
            ),

            "student_answer": str(
                row["student_answer"]
            ),

            "correct_answer": str(
                row["correct_answer"]
            ),

            "topic": str(
                row["topic"]
            )
        })

    return wrong_question_list


# ============================================================
# 12. PRIMARY WEAK TOPIC
# ============================================================

def get_primary_weak_topic(
    weak_topics_df
):

    if weak_topics_df.empty:

        return "General Concepts"

    return str(
        weak_topics_df.iloc[0]["topic"]
    )


# ============================================================
# 13. LOAD OLLAMA / AIPIPE
# ============================================================

load_dotenv(
    PROJECT_ROOT / ".env"
)

from aipipe import LLM

config_file = (
    PROJECT_ROOT /
    "config" /
    "llm.yaml"
)

if not config_file.exists():

    raise FileNotFoundError(
        f"LLM config not found: {config_file}"
    )

llm = LLM.from_config(
    config_file
)


# ============================================================
# 14. BUILD PERSONALIZED PROMPT
# ============================================================

def build_personalized_prompt(
    student,
    overall_score,
    risk_level,
    primary_topic,
    wrong_question_list
):

    wrong_questions_json = json.dumps(
        wrong_question_list,
        indent=2,
        default=str
    )

    return f"""
You are a Personalized AI Learning Assistant
inside an LMS.

Analyze the student's quiz performance and create
a personalized learning recommendation.

STUDENT
-------
Name: {student["name"]}
Overall Score: {overall_score:.2f}%
Risk Level: {risk_level}

WEAK TOPIC
----------
{primary_topic}

WRONG QUESTIONS
---------------
{wrong_questions_json}

TASK
----
1. Identify the main learning gap.
2. Explain the weak topic in very simple language.
3. Explain it like teaching a 5-year-old.
4. Explain the likely misunderstanding.
5. Give a real-world analogy.
6. Give a simple coding example if applicable.
7. Recommend the lesson/topic to study.
8. Give one practical activity.
9. Generate 3 practice questions.
10. Give a short motivational message.

IMPORTANT
---------
Focus only on the student's weak topic.
Do not recommend unrelated topics.

Return ONLY valid JSON.

FORMAT
------
{{
    "student": "",
    "risk_level": "",
    "weak_topic": "",
    "learning_gap": "",
    "simple_explanation": "",
    "five_year_old_explanation": "",
    "real_world_analogy": "",
    "coding_example": "",
    "recommended_lesson": "",
    "practice_activity": "",
    "practice_questions": [],
    "motivation": ""
}}
"""


# ============================================================
# 15. GENERATE AI RESPONSE
# ============================================================

def generate_personalized_response(
    llm,
    prompt
):

    return llm.generate(prompt)


# ============================================================
# 16. EXTRACT AI TEXT
# ============================================================

def extract_ai_text(
    ai_response
):

    if hasattr(
        ai_response,
        "content"
    ):

        return ai_response.content

    elif hasattr(
        ai_response,
        "text"
    ):

        return ai_response.text

    elif hasattr(
        ai_response,
        "response"
    ):

        return ai_response.response

    return str(ai_response)


# ============================================================
# 17. PARSE AI JSON
# ============================================================

def parse_ai_json(
    ai_text
):

    cleaned_text = ai_text.strip()

    cleaned_text = re.sub(
        r"^```json\s*",
        "",
        cleaned_text,
        flags=re.IGNORECASE
    )

    cleaned_text = re.sub(
        r"\s*```$",
        "",
        cleaned_text
    )

    cleaned_text = cleaned_text.strip()

    try:

        return json.loads(
            cleaned_text
        )

    except json.JSONDecodeError as e:

        print(
            "❌ JSON parsing failed"
        )

        print("Error:", e)

        print(
            "\nAI Response:"
        )

        print(cleaned_text)

        return None


# ============================================================
# 18. DISPLAY RESULT
# ============================================================

def display_personalized_result(
    result
):

    if not result:

        print(
            "❌ No personalized result available"
        )

        return

    print("=" * 60)
    print(
        "PERSONALIZED LEARNING RECOMMENDATION"
    )
    print("=" * 60)

    print(
        "\n🎯 Weak Topic:"
    )

    print(
        result.get("weak_topic")
    )

    print(
        "\n📌 Learning Gap:"
    )

    print(
        result.get("learning_gap")
    )

    print(
        "\n📖 Simple Explanation:"
    )

    print(
        result.get("simple_explanation")
    )

    print(
        "\n👶 5-Year-Old Explanation:"
    )

    print(
        result.get(
            "five_year_old_explanation"
        )
    )

    print(
        "\n🌍 Real-World Analogy:"
    )

    print(
        result.get(
            "real_world_analogy"
        )
    )

    print(
        "\n💻 Coding Example:"
    )

    print(
        result.get(
            "coding_example"
        )
    )

    print(
        "\n📚 Recommended Lesson:"
    )

    print(
        result.get(
            "recommended_lesson"
        )
    )

    print(
        "\n🧪 Practice Activity:"
    )

    print(
        result.get(
            "practice_activity"
        )
    )

    print(
        "\n📝 Practice Questions:"
    )

    for i, question in enumerate(
        result.get(
            "practice_questions",
            []
        ),
        1
    ):

        print(
            f"{i}. {question}"
        )

    print(
        "\n💪 Motivation:"
    )

    print(
        result.get(
            "motivation"
        )
    )


print("✅ All helper functions loaded successfully")

✅ All helper functions loaded successfully


In [9]:
# ============================================================
# TEST
# ============================================================

print("get_student:", "get_student" in globals())
print("get_quiz_attempts:", "get_quiz_attempts" in globals())
print("get_question_analysis:", "get_question_analysis" in globals())
print("build_personalized_prompt:", "build_personalized_prompt" in globals())
print("display_personalized_result:", "display_personalized_result" in globals())

get_student: True
get_quiz_attempts: True
get_question_analysis: True
build_personalized_prompt: True
display_personalized_result: True


In [10]:
result = run_personalized_ai(
    student_id=1
)

STARTING PERSONALIZED AI ANALYSIS

👤 Student: Demo Student
📊 Quiz Attempts: 6
📈 Overall Score: 66.67%
⚠️ Risk Level: medium
📝 Questions Analyzed: 30
❌ Wrong Questions: 4
🎯 Weak Topics Found: 1
📚 Primary Weak Topic: Variables

🤖 Generating personalized recommendation...

✅ Personalized AI analysis completed!


In [11]:
display_personalized_result(
    result["ai_result"]
)

PERSONALIZED LEARNING RECOMMENDATION

🎯 Weak Topic:
Variables

📌 Learning Gap:
The student is struggling with the fundamental concept of variables, specifically how they store and retrieve information, as evidenced by multiple incorrect answers across different quizzes.

📖 Simple Explanation:
A variable is like a labeled container that holds a piece of information. Instead of remembering the information itself, you just remember the name of the container to get the value inside.

👶 5-Year-Old Explanation:
Imagine you have a bunch of toy boxes. You put a teddy bear in one box and write 'My Favorite Toy' on the outside. Whenever you want your teddy bear, you just look for the box that says 'My Favorite Toy'!

🌍 Real-World Analogy:
Think of a variable like a contact name in your phone. You don't memorize the long phone number; you just save it under the name 'Mom'. When you click 'Mom', the phone knows exactly which number to call.

💻 Coding Example:
let playerName = 'Demo Student';
let p

In [12]:
# ============================================================
# CELL 26 - Save Personalized AI Analysis
# ============================================================

def save_ai_analysis(result):

    if result is None:
        print("❌ No result to save")
        return None

    ai_result = result.get("ai_result")

    if not ai_result:
        print("❌ AI result is empty")
        return None

    student = result["student"]

    # Get latest quiz attempt
    quiz_attempts = result["quiz_attempts"]

    if quiz_attempts.empty:
        print("❌ No quiz attempt found")
        return None

    latest_attempt = quiz_attempts.iloc[-1]

    attempt_id = int(
        latest_attempt["attempt_id"]
    )

    quiz_id = int(
        latest_attempt["quiz_id"]
    )

    score = float(
        result["overall_score"]
    )

    total_questions = int(
        latest_attempt["total_questions"]
    )

    correct_questions = int(
        latest_attempt["correct_questions"]
    )

    wrong_questions = (
        total_questions -
        correct_questions
    )

    # Weak topics
    weak_topics = []

    for _, row in result["weak_topics"].iterrows():

        weak_topics.append({
            "topic_id": int(row["topic_id"]),
            "topic": str(row["topic"]),
            "accuracy": float(row["accuracy"])
        })

    # Question analysis
    question_analysis = []

    for _, row in result["wrong_questions"].iterrows():

        question_analysis.append({
            "question_id": int(row["question_id"]),
            "question": str(row["question_text"]),
            "student_answer": str(row["student_answer"]),
            "correct_answer": str(row["correct_answer"]),
            "topic": str(row["topic"])
        })

    # Recommendations
    recommendations = {
        "weak_topic": ai_result.get("weak_topic"),
        "recommended_lesson": ai_result.get(
            "recommended_lesson"
        ),
        "practice_activity": ai_result.get(
            "practice_activity"
        ),
        "practice_questions": ai_result.get(
            "practice_questions",
            []
        ),
        "motivation": ai_result.get(
            "motivation"
        )
    }

    insert_query = text("""
        INSERT INTO ai_quiz_analysis (
            student_id,
            attempt_id,
            quiz_id,
            score,
            total_questions,
            correct_questions,
            wrong_questions,
            weak_topics,
            question_analysis,
            ai_summary,
            recommendations
        )
        VALUES (
            :student_id,
            :attempt_id,
            :quiz_id,
            :score,
            :total_questions,
            :correct_questions,
            :wrong_questions,
            :weak_topics,
            :question_analysis,
            :ai_summary,
            :recommendations
        )
    """)

    with engine.begin() as connection:

        result_db = connection.execute(
            insert_query,
            {
                "student_id": int(
                    student["id"]
                ),

                "attempt_id": attempt_id,

                "quiz_id": quiz_id,

                "score": score,

                "total_questions": total_questions,

                "correct_questions": correct_questions,

                "wrong_questions": wrong_questions,

                "weak_topics": json.dumps(
                    weak_topics
                ),

                "question_analysis": json.dumps(
                    question_analysis
                ),

                "ai_summary": ai_result.get(
                    "learning_gap"
                ),

                "recommendations": json.dumps(
                    recommendations
                )
            }
        )

    print(
        "✅ Personalized AI analysis saved to database"
    )

    return result_db.lastrowid

In [13]:
# ============================================================
# CELL 27 - Execute Save
# ============================================================

analysis_id = save_ai_analysis(
    result
)

print(
    "Analysis ID:",
    analysis_id
)

✅ Personalized AI analysis saved to database
Analysis ID: 1


In [14]:
# ============================================================
# CELL 28 - Verify Saved Analysis
# ============================================================

def get_saved_ai_analysis(
    student_id
):

    query = text("""
        SELECT
            id,
            student_id,
            attempt_id,
            quiz_id,
            score,
            total_questions,
            correct_questions,
            wrong_questions,
            weak_topics,
            question_analysis,
            ai_summary,
            recommendations,
            created_at
        FROM ai_quiz_analysis
        WHERE student_id = :student_id
        ORDER BY created_at DESC
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query,
            {
                "student_id": student_id
            }
        ).mappings().all()

    return pd.DataFrame(rows)

In [15]:
saved_analysis_df = get_saved_ai_analysis(1)

display(saved_analysis_df)

,ai_summary,attempt_id,correct_questions,created_at,id,question_analysis,quiz_id,recommendations,score,student_id,total_questions,weak_topics,wrong_questions
0,The student is struggling with the fundamental...,1,4,2026-09-23 15:04:19,1,"[{""topic"": ""Variables"", ""question"": ""Question ...",1,"{""motivation"": ""You're doing great! Mastering ...",66.67,1,5,"[{""topic"": ""Variables"", ""accuracy"": 40.0, ""top...",1


In [16]:
# ============================================================
# OVERALL COURSE ANALYSIS
# ============================================================

def get_course_overall_analysis():

    query = text("""
        SELECT
            COUNT(DISTINCT qa.student_id) AS total_students,
            COUNT(DISTINCT qa.id) AS total_attempts,

            SUM(qa.total_questions) AS total_questions,
            SUM(qa.correct_questions) AS total_correct,

            AVG(qa.score) AS average_score,

            MIN(qa.score) AS minimum_score,
            MAX(qa.score) AS maximum_score

        FROM quiz_attempts qa
    """)

    with engine.connect() as connection:

        result = connection.execute(query).mappings().first()

    if result is None:
        return {}

    data = dict(result)

    total_questions = data["total_questions"] or 0
    total_correct = data["total_correct"] or 0

    if total_questions > 0:
        overall_accuracy = (
            total_correct /
            total_questions
        ) * 100
    else:
        overall_accuracy = 0

    data["overall_accuracy"] = round(
        overall_accuracy,
        2
    )

    return data

In [17]:
course_overall = get_course_overall_analysis()

print(course_overall)

{'total_students': 1, 'total_attempts': 6, 'total_questions': Decimal('30'), 'total_correct': Decimal('20'), 'average_score': Decimal('63.333333'), 'minimum_score': Decimal('40.00'), 'maximum_score': Decimal('80.00'), 'overall_accuracy': Decimal('66.67')}


In [ ]:
# Student performance across the entire course
def get_all_student_performance():

    query = text("""
        SELECT
            s.id AS student_id,
            s.name AS student_name,
            s.batch,

            COUNT(qa.id) AS attempts,

            SUM(qa.total_questions) AS total_questions,

            SUM(qa.correct_questions) AS correct_questions,

            AVG(qa.score) AS average_score

        FROM students s

        LEFT JOIN quiz_attempts qa
            ON s.id = qa.student_id

        GROUP BY
            s.id,
            s.name,
            s.batch

        ORDER BY average_score DESC
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query
        ).mappings().all()

    df = pd.DataFrame(rows)

    if not df.empty:

        df["average_score"] = (
            df["average_score"]
            .fillna(0)
            .round(2)
        )

        df["risk_level"] = df[
            "average_score"
        ].apply(
            calculate_risk_level
        )

    return df

In [20]:
student_course_df = (
    get_all_student_performance()
)

display(student_course_df)

,attempts,average_score,batch,correct_questions,student_id,student_name,total_questions,risk_level
0,6,63.333333,Aarohan-2026,20,1,Demo Student,30,medium
1,0,0,Aarohan-2026,None,2,Demo Student 2,None,high
2,0,0,Aarohan-2026,None,3,Demo Student 3,None,high


In [21]:
# At-risk students
def get_at_risk_students(
    student_course_df
):

    if student_course_df.empty:
        return pd.DataFrame()

    return student_course_df[
        student_course_df["risk_level"].isin(
            ["medium", "high"]
        )
    ].copy()
at_risk_students = get_at_risk_students(
    student_course_df
)

display(at_risk_students)

,attempts,average_score,batch,correct_questions,student_id,student_name,total_questions,risk_level
0,6,63.333333,Aarohan-2026,20,1,Demo Student,30,medium
1,0,0,Aarohan-2026,None,2,Demo Student 2,None,high
2,0,0,Aarohan-2026,None,3,Demo Student 3,None,high


In [22]:
# Module-wise performance
def get_module_performance():

    query = text("""
        SELECT

            m.id AS module_id,

            m.name AS module_name,

            COUNT(DISTINCT qa.student_id)
                AS students,

            COUNT(qa.id)
                AS attempts,

            SUM(qa.total_questions)
                AS total_questions,

            SUM(qa.correct_questions)
                AS correct_questions,

            AVG(qa.score)
                AS average_score

        FROM quiz_attempts qa

        JOIN quizzes q
            ON qa.quiz_id = q.id

        JOIN days d
            ON q.day_id = d.id

        JOIN modules m
            ON d.module_id = m.id

        GROUP BY
            m.id,
            m.name

        ORDER BY
            m.id
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query
        ).mappings().all()

    df = pd.DataFrame(rows)

    if not df.empty:

        df["average_score"] = (
            df["average_score"]
            .round(2)
        )

        df["accuracy"] = (
            df["correct_questions"]
            /
            df["total_questions"]
            * 100
        ).round(2)

    return df
module_performance_df = (
    get_module_performance()
)

display(module_performance_df)

,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy
0,6,63.333333,20,1,Python Basics,1,30,66.66666666666666666666666667


In [23]:
# Day-wise performance
def get_day_performance():

    query = text("""
        SELECT

            m.id AS module_id,
            m.name AS module_name,

            d.id AS day_id,
            d.day_number,

            COUNT(DISTINCT qa.student_id)
                AS students,

            COUNT(qa.id)
                AS attempts,

            SUM(qa.total_questions)
                AS total_questions,

            SUM(qa.correct_questions)
                AS correct_questions,

            AVG(qa.score)
                AS average_score

        FROM quiz_attempts qa

        JOIN quizzes q
            ON qa.quiz_id = q.id

        JOIN days d
            ON q.day_id = d.id

        JOIN modules m
            ON d.module_id = m.id

        GROUP BY
            m.id,
            m.name,
            d.id,
            d.day_number

        ORDER BY
            m.id,
            d.day_number
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query
        ).mappings().all()

    df = pd.DataFrame(rows)

    if not df.empty:

        df["average_score"] = (
            df["average_score"]
            .round(2)
        )

        df["accuracy"] = (
            df["correct_questions"]
            /
            df["total_questions"]
            * 100
        ).round(2)

    return df

In [24]:
day_performance_df = (
    get_day_performance()
)

display(day_performance_df)

,attempts,average_score,correct_questions,day_id,day_number,module_id,module_name,students,total_questions,accuracy
0,1,40.000000,2,6,1,1,Python Basics,1,5,40.0
1,1,60.000000,3,5,2,1,Python Basics,1,5,60.0
2,1,55.000000,3,4,3,1,Python Basics,1,5,60.0
3,1,70.000000,4,3,4,1,Python Basics,1,5,80.0
4,1,75.000000,4,2,5,1,Python Basics,1,5,80.0
5,1,80.000000,4,1,6,1,Python Basics,1,5,80.0


In [27]:
# ============================================================
# STEP 6 - COURSE TOPIC PERFORMANCE
# ============================================================

def get_course_topic_performance():

    query = text("""
        SELECT
            t.id AS topic_id,
            t.name AS topic,

            COUNT(DISTINCT qa.id) AS total_questions,

            SUM(
                CASE
                    WHEN qa.is_correct = 1
                    THEN 1
                    ELSE 0
                END
            ) AS correct_answers,

            SUM(
                CASE
                    WHEN qa.is_correct = 0
                    THEN 1
                    ELSE 0
                END
            ) AS wrong_answers

        FROM question_attempts qa

        INNER JOIN questions q
            ON qa.question_id = q.id

        INNER JOIN topics t
            ON q.topic_id = t.id

        GROUP BY
            t.id,
            t.name

        ORDER BY
            t.id
    """)

    try:

        with engine.connect() as connection:

            rows = connection.execute(
                query
            ).mappings().all()

        df = pd.DataFrame(rows)

        if df.empty:

            print("⚠️ No topic performance data found")

            return df

        # Calculate accuracy
        df["accuracy"] = (
            df["correct_answers"]
            /
            df["total_questions"]
            * 100
        ).round(2)

        return df

    except Exception as e:

        print(
            "❌ Error while getting course topic performance:"
        )

        print(e)

        return pd.DataFrame()

In [28]:
course_topic_df = get_course_topic_performance()

display(course_topic_df)

,correct_answers,topic,topic_id,total_questions,wrong_answers,accuracy
0,5,Data Types,1,5,0,100
1,5,Functions,2,5,0,100
2,5,Variables,3,5,0,100
3,4,Data Types,4,5,1,80.0
4,5,Functions,5,5,0,100
5,2,Variables,6,5,3,40.0


In [30]:
# Identify course-wide weak topics
def identify_course_weak_topics(
    course_topic_df,
    threshold=70
):

    if course_topic_df.empty:
        return pd.DataFrame()

    return (
        course_topic_df[
            course_topic_df["accuracy"] < threshold
        ]
        .sort_values(
            "accuracy"
        )
        .copy()
    )

In [31]:
course_weak_topics_df = (
    identify_course_weak_topics(
        course_topic_df
    )
)

display(course_weak_topics_df)

,correct_answers,topic,topic_id,total_questions,wrong_answers,accuracy
5,2,Variables,6,5,3,40.0


In [32]:
# Create one overall course function
def run_course_analysis():

    print("=" * 70)
    print("OVERALL COURSE ANALYSIS")
    print("=" * 70)

    # Overall
    overall = (
        get_course_overall_analysis()
    )

    # Students
    students = (
        get_all_student_performance()
    )

    # At risk
    at_risk = (
        get_at_risk_students(
            students
        )
    )

    # Modules
    modules = (
        get_module_performance()
    )

    # Days
    days = (
        get_day_performance()
    )

    # Topics
    topics = (
        get_course_topic_performance()
    )

    # Weak topics
    weak_topics = (
        identify_course_weak_topics(
            topics
        )
    )

    return {
        "overall": overall,
        "students": students,
        "at_risk_students": at_risk,
        "modules": modules,
        "days": days,
        "topics": topics,
        "weak_topics": weak_topics
    }

In [33]:
course_result = run_course_analysis()

OVERALL COURSE ANALYSIS


In [35]:
# Display the overall course analysis
print("=" * 70)
print("COURSE SUMMARY")
print("=" * 70)

print(
    "Total Students:",
    course_result["overall"]["total_students"]
)

print(
    "Total Quiz Attempts:",
    course_result["overall"]["total_attempts"]
)

print(
    "Average Score:",
    round(
        float(
            course_result["overall"]["average_score"]
        ),
        2
    ),
    "%"
)

print(
    "Overall Accuracy:",
    course_result["overall"]["overall_accuracy"],
    "%"
)

COURSE SUMMARY
Total Students: 1
Total Quiz Attempts: 6
Average Score: 63.33 %
Overall Accuracy: 66.67 %


In [36]:
print("\nMODULE PERFORMANCE")
display(
    course_result["modules"]
)


MODULE PERFORMANCE


,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy
0,6,63.333333,20,1,Python Basics,1,30,66.66666666666666666666666667


In [37]:
print("\nWEAK TOPICS")
display(
    course_result["weak_topics"]
    
)


WEAK TOPICS


,correct_answers,topic,topic_id,total_questions,wrong_answers,accuracy
5,2,Variables,6,5,3,40.0


In [38]:
print("\nAT-RISK STUDENTS")
display(
    course_result["at_risk_students"]
)


AT-RISK STUDENTS


,attempts,average_score,batch,correct_questions,student_id,student_name,total_questions,risk_level
0,6,63.333333,Aarohan-2026,20,1,Demo Student,30,medium
1,0,0,Aarohan-2026,None,2,Demo Student 2,None,high
2,0,0,Aarohan-2026,None,3,Demo Student 3,None,high


In [39]:
# ============================================================
# MODULE-WISE COURSE PERFORMANCE
# ============================================================

def get_module_performance():

    query = text("""
        SELECT

            m.id AS module_id,

            m.name AS module_name,

            COUNT(DISTINCT qa.student_id)
                AS students,

            COUNT(qa.id)
                AS attempts,

            SUM(qa.total_questions)
                AS total_questions,

            SUM(qa.correct_questions)
                AS correct_questions,

            AVG(qa.score)
                AS average_score

        FROM quiz_attempts qa

        INNER JOIN quizzes q
            ON qa.quiz_id = q.id

        INNER JOIN days d
            ON q.day_id = d.id

        INNER JOIN modules m
            ON d.module_id = m.id

        GROUP BY
            m.id,
            m.name

        ORDER BY
            m.id
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query
        ).mappings().all()

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    # Calculate question accuracy
    df["accuracy"] = (
        df["correct_questions"]
        /
        df["total_questions"]
        * 100
    ).round(2)

    # Average quiz score
    df["average_score"] = (
        df["average_score"]
        .round(2)
    )

    # Module status
    df["status"] = df["accuracy"].apply(
        assign_module_status
    )

    return df

In [40]:
# ============================================================
# MODULE STATUS
# ============================================================

def assign_module_status(accuracy):

    if pd.isna(accuracy):
        return "No Data"

    if accuracy < 50:
        return "Needs Immediate Attention"

    elif accuracy < 70:
        return "Needs Improvement"

    elif accuracy < 85:
        return "Good"

    else:
        return "Strong"

In [41]:
module_performance_df = (
    get_module_performance()
)

display(module_performance_df)

,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy,status
0,6,63.333333,20,1,Python Basics,1,30,66.66666666666666666666666667,Needs Improvement


In [42]:
# ============================================================
# MODULE ANALYSIS
# ============================================================

def get_module_summary(module_df):

    if module_df.empty:
        return {
            "strong_modules": pd.DataFrame(),
            "weak_modules": pd.DataFrame()
        }

    weak_modules = module_df[
        module_df["accuracy"] < 70
    ].copy()

    strong_modules = module_df[
        module_df["accuracy"] >= 85
    ].copy()

    return {
        "strong_modules": strong_modules,
        "weak_modules": weak_modules
    }

In [43]:
module_summary = get_module_summary(
    module_performance_df
)

print("🔴 Modules Needing Improvement")
display(
    module_summary["weak_modules"]
)

print("\n🟢 Strong Modules")
display(
    module_summary["strong_modules"]
)

🔴 Modules Needing Improvement


,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy,status
0,6,63.333333,20,1,Python Basics,1,30,66.66666666666666666666666667,Needs Improvement



🟢 Strong Modules


,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy,status


In [44]:
# ============================================================
# DAY-WISE COURSE PERFORMANCE
# ============================================================

def get_day_performance():

    query = text("""
        SELECT

            m.id AS module_id,

            m.name AS module_name,

            d.id AS day_id,

            d.day_number,

            COUNT(DISTINCT qa.student_id)
                AS students,

            COUNT(qa.id)
                AS attempts,

            SUM(qa.total_questions)
                AS total_questions,

            SUM(qa.correct_questions)
                AS correct_questions,

            AVG(qa.score)
                AS average_score

        FROM quiz_attempts qa

        INNER JOIN quizzes q
            ON qa.quiz_id = q.id

        INNER JOIN days d
            ON q.day_id = d.id

        INNER JOIN modules m
            ON d.module_id = m.id

        GROUP BY
            m.id,
            m.name,
            d.id,
            d.day_number

        ORDER BY
            m.id,
            d.day_number
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query
        ).mappings().all()

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df["average_score"] = (
        df["average_score"]
        .round(2)
    )

    df["accuracy"] = (
        df["correct_questions"]
        /
        df["total_questions"]
        * 100
    ).round(2)

    df["status"] = df["accuracy"].apply(
        assign_module_status
    )

    return df

In [45]:
day_performance_df = get_day_performance()

display(day_performance_df)

,attempts,average_score,correct_questions,day_id,day_number,module_id,module_name,students,total_questions,accuracy,status
0,1,40.000000,2,6,1,1,Python Basics,1,5,40.0,Needs Immediate Attention
1,1,60.000000,3,5,2,1,Python Basics,1,5,60.0,Needs Improvement
2,1,55.000000,3,4,3,1,Python Basics,1,5,60.0,Needs Improvement
3,1,70.000000,4,3,4,1,Python Basics,1,5,80.0,Good
4,1,75.000000,4,2,5,1,Python Basics,1,5,80.0,Good
5,1,80.000000,4,1,6,1,Python Basics,1,5,80.0,Good


In [46]:
# ============================================================
# TOPIC-WISE COURSE PERFORMANCE
# ============================================================

def get_course_topic_performance():

    query = text("""
        SELECT

            t.id AS topic_id,

            t.name AS topic,

            COUNT(DISTINCT qa.student_id)
                AS students,

            COUNT(qa.id)
                AS total_questions,

            SUM(
                CASE
                    WHEN qa.is_correct = 1
                    THEN 1
                    ELSE 0
                END
            ) AS correct_answers,

            SUM(
                CASE
                    WHEN qa.is_correct = 0
                    THEN 1
                    ELSE 0
                END
            ) AS wrong_answers

        FROM question_attempts qa

        INNER JOIN questions q
            ON qa.question_id = q.id

        INNER JOIN topics t
            ON q.topic_id = t.id

        GROUP BY
            t.id,
            t.name

        ORDER BY
            t.id
    """)

    with engine.connect() as connection:

        rows = connection.execute(
            query
        ).mappings().all()

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df["accuracy"] = (
        df["correct_answers"]
        /
        df["total_questions"]
        * 100
    ).round(2)

    df["status"] = df["accuracy"].apply(
        assign_module_status
    )

    return df

In [48]:
# Replace your topic-performance cell
from sqlalchemy import text
import pandas as pd

def get_course_topic_performance():

    query = text("""
        SELECT
            t.id AS topic_id,
            t.name AS topic,

            COUNT(qa.id) AS total_questions,

            SUM(
                CASE
                    WHEN qa.is_correct = 1 THEN 1
                    ELSE 0
                END
            ) AS correct_answers,

            SUM(
                CASE
                    WHEN qa.is_correct = 0 THEN 1
                    ELSE 0
                END
            ) AS wrong_answers

        FROM question_attempts qa

        INNER JOIN questions q
            ON qa.question_id = q.id

        INNER JOIN topics t
            ON q.topic_id = t.id

        GROUP BY
            t.id,
            t.name

        ORDER BY
            t.id
    """)

    with engine.connect() as conn:
        df = pd.read_sql(query, conn)

    # Make sure numeric columns are numeric
    df["total_questions"] = pd.to_numeric(
        df["total_questions"], errors="coerce"
    ).fillna(0)

    df["correct_answers"] = pd.to_numeric(
        df["correct_answers"], errors="coerce"
    ).fillna(0)

    df["wrong_answers"] = pd.to_numeric(
        df["wrong_answers"], errors="coerce"
    ).fillna(0)

    # Calculate accuracy
    df["accuracy"] = (
        df["correct_answers"]
        / df["total_questions"]
        * 100
    ).round(2)

    # Status
    def topic_status(accuracy):
        if accuracy < 50:
            return "Needs Immediate Attention"
        elif accuracy < 70:
            return "Needs Improvement"
        elif accuracy < 85:
            return "Good"
        else:
            return "Strong"

    df["status"] = df["accuracy"].apply(topic_status)

    return df

In [49]:
course_topic_df = get_course_topic_performance()

course_topic_df

,topic_id,topic,total_questions,correct_answers,wrong_answers,accuracy,status
0,1,Data Types,5,5.0,0.0,100.0,Strong
1,2,Functions,5,5.0,0.0,100.0,Strong
2,3,Variables,5,5.0,0.0,100.0,Strong
3,4,Data Types,5,4.0,1.0,80.0,Good
4,5,Functions,5,5.0,0.0,100.0,Strong
5,6,Variables,5,2.0,3.0,40.0,Needs Immediate Attention


In [50]:
# Check weak topics
weak_topics_df = course_topic_df[
    course_topic_df["accuracy"] < 70
].copy()

weak_topics_df

,topic_id,topic,total_questions,correct_answers,wrong_answers,accuracy,status
5,6,Variables,5,2.0,3.0,40.0,Needs Immediate Attention


In [51]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            t.id,
            t.name
        FROM topics t
        ORDER BY t.id
    """))

    print(result.fetchall())

[(1, 'Data Types'), (2, 'Functions'), (3, 'Variables'), (4, 'Data Types'), (5, 'Functions'), (6, 'Variables'), (7, 'Data Types'), (8, 'Functions'), (9, 'Variables'), (10, 'Data Types'), (11, 'Functions'), (12, 'Variables'), (13, 'Data Types'), (14, 'Functions'), (15, 'Variables'), (16, 'Data Types'), (17, 'Functions'), (18, 'Variables'), (19, 'Data Types'), (20, 'Functions'), (21, 'Variables'), (22, 'Data Types'), (23, 'Functions'), (24, 'Variables'), (25, 'Data Types'), (26, 'Functions'), (27, 'Variables'), (28, 'Data Types'), (29, 'Functions'), (30, 'Variables'), (31, 'Data Types'), (32, 'Functions'), (33, 'Variables'), (34, 'Data Types'), (35, 'Functions'), (36, 'Variables'), (37, 'Data Types'), (38, 'Functions'), (39, 'Variables'), (40, 'Data Types'), (41, 'Functions'), (42, 'Variables'), (43, 'Data Types'), (44, 'Functions'), (45, 'Variables'), (46, 'Data Types'), (47, 'Functions'), (48, 'Variables'), (49, 'Data Types'), (50, 'Functions'), (51, 'Variables'), (52, 'Data Types'), (5

In [52]:
# Create the final overall analysis function
def run_overall_course_analysis():

    print("=" * 70)
    print("OVERALL COURSE ANALYSIS")
    print("=" * 70)

    # ---------------------------------------------------------
    # 1. COURSE OVERALL
    # ---------------------------------------------------------
    course_overall = get_course_overall_analysis()

    print("\n📊 COURSE OVERALL")
    print("-" * 70)
    print(course_overall)

    # ---------------------------------------------------------
    # 2. STUDENT-WISE
    # ---------------------------------------------------------
    student_df = get_all_student_performance()

    # Do NOT convert missing scores to 0 before assigning status
    def assign_course_status(row):

        if pd.isna(row["average_score"]):
            return "not_assessed"

        score = float(row["average_score"])

        if score < 50:
            return "high"
        elif score < 70:
            return "medium"
        else:
            return "low"

    student_df["status"] = student_df.apply(
        assign_course_status,
        axis=1
    )

    print("\n👨‍🎓 STUDENT-WISE PERFORMANCE")
    print("-" * 70)
    display(student_df)

    # ---------------------------------------------------------
    # 3. MODULE-WISE
    # ---------------------------------------------------------
    module_df = get_module_performance()

    print("\n📚 MODULE-WISE PERFORMANCE")
    print("-" * 70)
    display(module_df)

    # ---------------------------------------------------------
    # 4. DAY-WISE
    # ---------------------------------------------------------
    day_df = get_day_performance()

    print("\n📅 DAY-WISE PERFORMANCE")
    print("-" * 70)
    display(day_df)

    # ---------------------------------------------------------
    # 5. TOPIC-WISE
    # ---------------------------------------------------------
    topic_df = get_course_topic_performance()

    print("\n🎯 TOPIC-WISE PERFORMANCE")
    print("-" * 70)
    display(topic_df)

    # ---------------------------------------------------------
    # 6. WEAK MODULES
    # ---------------------------------------------------------
    weak_modules = module_df[
        module_df["accuracy"] < 70
    ].copy()

    # ---------------------------------------------------------
    # 7. WEAK DAYS
    # ---------------------------------------------------------
    weak_days = day_df[
        day_df["accuracy"] < 70
    ].copy()

    # ---------------------------------------------------------
    # 8. WEAK TOPICS
    # ---------------------------------------------------------
    weak_topics = topic_df[
        topic_df["accuracy"] < 70
    ].copy()

    print("\n⚠️ WEAK MODULES")
    print("-" * 70)
    display(weak_modules)

    print("\n⚠️ WEAK DAYS")
    print("-" * 70)
    display(weak_days)

    print("\n⚠️ WEAK TOPICS")
    print("-" * 70)
    display(weak_topics)

    # ---------------------------------------------------------
    # 9. FINAL RESULT
    # ---------------------------------------------------------
    result = {
        "course_overall": course_overall,
        "student_performance": student_df,
        "module_performance": module_df,
        "day_performance": day_df,
        "topic_performance": topic_df,
        "weak_modules": weak_modules,
        "weak_days": weak_days,
        "weak_topics": weak_topics
    }

    return result

In [53]:
overall_result = run_overall_course_analysis()

OVERALL COURSE ANALYSIS

📊 COURSE OVERALL
----------------------------------------------------------------------
{'total_students': 1, 'total_attempts': 6, 'total_questions': Decimal('30'), 'total_correct': Decimal('20'), 'average_score': Decimal('63.333333'), 'minimum_score': Decimal('40.00'), 'maximum_score': Decimal('80.00'), 'overall_accuracy': Decimal('66.67')}

👨‍🎓 STUDENT-WISE PERFORMANCE
----------------------------------------------------------------------


,attempts,average_score,batch,correct_questions,student_id,student_name,total_questions,risk_level,status
0,6,63.333333,Aarohan-2026,20,1,Demo Student,30,medium,medium
1,0,0,Aarohan-2026,None,2,Demo Student 2,None,high,high
2,0,0,Aarohan-2026,None,3,Demo Student 3,None,high,high



📚 MODULE-WISE PERFORMANCE
----------------------------------------------------------------------


,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy,status
0,6,63.333333,20,1,Python Basics,1,30,66.66666666666666666666666667,Needs Improvement



📅 DAY-WISE PERFORMANCE
----------------------------------------------------------------------


,attempts,average_score,correct_questions,day_id,day_number,module_id,module_name,students,total_questions,accuracy,status
0,1,40.000000,2,6,1,1,Python Basics,1,5,40.0,Needs Immediate Attention
1,1,60.000000,3,5,2,1,Python Basics,1,5,60.0,Needs Improvement
2,1,55.000000,3,4,3,1,Python Basics,1,5,60.0,Needs Improvement
3,1,70.000000,4,3,4,1,Python Basics,1,5,80.0,Good
4,1,75.000000,4,2,5,1,Python Basics,1,5,80.0,Good
5,1,80.000000,4,1,6,1,Python Basics,1,5,80.0,Good


... (4 outputs left out)


⚠️ WEAK DAYS
----------------------------------------------------------------------


,attempts,average_score,correct_questions,day_id,day_number,module_id,module_name,students,total_questions,accuracy,status
0,1,40.000000,2,6,1,1,Python Basics,1,5,40.0,Needs Immediate Attention
1,1,60.000000,3,5,2,1,Python Basics,1,5,60.0,Needs Improvement
2,1,55.000000,3,4,3,1,Python Basics,1,5,60.0,Needs Improvement



⚠️ WEAK TOPICS
----------------------------------------------------------------------


,topic_id,topic,total_questions,correct_answers,wrong_answers,accuracy,status
5,6,Variables,5,2.0,3.0,40.0,Needs Immediate Attention


In [54]:
overall_result.keys()

dict_keys(['course_overall', 'student_performance', 'module_performance', 'day_performance', 'topic_performance', 'weak_modules', 'weak_days', 'weak_topics'])

In [59]:
print("WEAK MODULES")
display(overall_result["weak_modules"])

print("WEAK DAYS")
display(overall_result["weak_days"])

print("WEAK TOPICS")
display(overall_result["weak_topics"])

WEAK MODULES


,attempts,average_score,correct_questions,module_id,module_name,students,total_questions,accuracy,status
0,6,63.333333,20,1,Python Basics,1,30,66.66666666666666666666666667,Needs Improvement


WEAK DAYS


,attempts,average_score,correct_questions,day_id,day_number,module_id,module_name,students,total_questions,accuracy,status
0,1,40.000000,2,6,1,1,Python Basics,1,5,40.0,Needs Immediate Attention
1,1,60.000000,3,5,2,1,Python Basics,1,5,60.0,Needs Improvement
2,1,55.000000,3,4,3,1,Python Basics,1,5,60.0,Needs Improvement


WEAK TOPICS


,topic_id,topic,total_questions,correct_answers,wrong_answers,accuracy,status
5,6,Variables,5,2.0,3.0,40.0,Needs Immediate Attention


In [64]:
# Create the AI summary function
import json
import re


def dataframe_to_records(df, columns=None):
    """
    Convert DataFrame into JSON-safe records.
    """

    if df is None or df.empty:
        return []

    if columns:
        available_columns = [
            col for col in columns
            if col in df.columns
        ]
        data = df[available_columns]
    else:
        data = df

    # Replace NaN/NaT with None
    data = data.where(pd.notnull(data), None)

    return data.to_dict(orient="records")


def build_overall_course_ai_prompt(overall_result):

    course_overall = overall_result["course_overall"]

    student_df = overall_result["student_performance"]
    module_df = overall_result["module_performance"]
    day_df = overall_result["day_performance"]
    topic_df = overall_result["topic_performance"]

    weak_modules = overall_result["weak_modules"]
    weak_days = overall_result["weak_days"]
    weak_topics = overall_result["weak_topics"]

    # ---------------------------------------------------------
    # STUDENT DATA
    # ---------------------------------------------------------

    student_data = dataframe_to_records(
        student_df,
        [
            "student_id",
            "student_name",
            "batch",
            "attempts",
            "average_score",
            "status"
        ]
    )

    # ---------------------------------------------------------
    # MODULE DATA
    # ---------------------------------------------------------

    module_data = dataframe_to_records(
        module_df,
        [
            "module_id",
            "module_name",
            "students",
            "attempts",
            "average_score",
            "accuracy",
            "status"
        ]
    )

    # ---------------------------------------------------------
    # DAY DATA
    # ---------------------------------------------------------

    day_data = dataframe_to_records(
        day_df,
        [
            "module_name",
            "day_number",
            "students",
            "attempts",
            "average_score",
            "accuracy",
            "status"
        ]
    )

    # ---------------------------------------------------------
    # TOPIC DATA
    # ---------------------------------------------------------

    topic_data = dataframe_to_records(
        topic_df,
        [
            "topic_id",
            "topic",
            "total_questions",
            "correct_answers",
            "wrong_answers",
            "accuracy",
            "status"
        ]
    )

    # ---------------------------------------------------------
    # WEAK AREAS
    # ---------------------------------------------------------

    weak_module_data = dataframe_to_records(
        weak_modules,
        [
            "module_id",
            "module_name",
            "average_score",
            "accuracy",
            "status"
        ]
    )

    weak_day_data = dataframe_to_records(
        weak_days,
        [
            "module_name",
            "day_number",
            "average_score",
            "accuracy",
            "status"
        ]
    )

    weak_topic_data = dataframe_to_records(
        weak_topics,
        [
            "topic_id",
            "topic",
            "total_questions",
            "correct_answers",
            "wrong_answers",
            "accuracy",
            "status"
        ]
    )

    prompt = f"""
You are an AI academic course analysis assistant.

Analyze the following LMS course performance data.

Your task is to help faculty understand the current course performance
and decide what teaching areas require attention.

IMPORTANT:
- Use ONLY the data provided below.
- Do not invent student performance.
- Do not invent modules or topics.
- Do not change numerical values.
- Students with no attempts are "not_assessed", not automatically weak.
- Keep recommendations practical and related to the observed data.

COURSE OVERALL:
{json.dumps(course_overall, indent=2, default=str)}

STUDENT-WISE PERFORMANCE:
{json.dumps(student_data, indent=2, default=str)}

MODULE-WISE PERFORMANCE:
{json.dumps(module_data, indent=2, default=str)}

DAY-WISE PERFORMANCE:
{json.dumps(day_data, indent=2, default=str)}

TOPIC-WISE PERFORMANCE:
{json.dumps(topic_data, indent=2, default=str)}

WEAK MODULES:
{json.dumps(weak_module_data, indent=2, default=str)}

WEAK DAYS:
{json.dumps(weak_day_data, indent=2, default=str)}

WEAK TOPICS:
{json.dumps(weak_topic_data, indent=2, default=str)}

Return ONLY valid JSON.

Use exactly this structure:

{{
    "course_summary": "",
    "overall_performance": "",
    "key_findings": [
        "",
        "",
        ""
    ],
    "student_observations": [
        ""
    ],
    "weak_modules": [
        {{
            "module": "",
            "accuracy": 0,
            "observation": ""
        }}
    ],
    "weak_topics": [
        {{
            "topic": "",
            "accuracy": 0,
            "observation": ""
        }}
    ],
    "teaching_recommendations": [
        "",
        "",
        ""
    ],
    "revision_priorities": [
        "",
        "",
        ""
    ],
    "faculty_action_plan": [
        "",
        "",
        ""
    ]
}}
"""

    return prompt

In [65]:
# Generate the Overall AI Analysis
def generate_overall_course_ai(overall_result):

    prompt = build_overall_course_ai_prompt(overall_result)

    response = llm.generate(prompt)

    ai_text = extract_ai_text(response)

    return ai_text

In [66]:
# Parse Ollama JSON
def parse_overall_ai_json(ai_text):

    if not ai_text:
        return None

    text = ai_text.strip()

    # Remove markdown code fences
    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    try:
        return json.loads(text)

    except json.JSONDecodeError as e:

        print("JSON parsing failed.")
        print("Error:", e)
        print("\nAI Response:")
        print(ai_text)

        return None

In [69]:
# Run the AI 
overall_ai_text = generate_overall_course_ai(
    overall_result
)


print(overall_ai_text)

```json
{
    "course_summary": "The course currently has 3 students enrolled in the Aarohan-2026 batch, with only 1 student actively participating in assessments for the 'Python Basics' module.",
    "overall_performance": "The overall accuracy is 66.67% with an average score of 63.33%. Performance shows a positive upward trend over the 6-day period, starting from a minimum score of 40.00 and reaching a maximum of 80.00.",
    "key_findings": [
        "There is a significant participation gap, as 2 out of 3 students have 0 attempts and are not assessed.",
        "Performance improved steadily from Day 1 (40% accuracy) to Day 6 (80% accuracy).",
        "While most topics are strong, there is a specific failure point in one of the 'Variables' topic assessments."
    ],
    "student_observations": [
        "Demo Student is the only active participant with 6 attempts and a 'medium' status.",
        "Demo Student 2 and Demo Student 3 are not_assessed."
    ],
    "weak_modules": [
   

In [70]:
overall_ai_result = parse_overall_ai_json(
    overall_ai_text
)


In [71]:
overall_ai_result

{'course_summary': "The course currently has 3 students enrolled in the Aarohan-2026 batch, with only 1 student actively participating in assessments for the 'Python Basics' module.",
 'overall_performance': 'The overall accuracy is 66.67% with an average score of 63.33%. Performance shows a positive upward trend over the 6-day period, starting from a minimum score of 40.00 and reaching a maximum of 80.00.',
 'key_findings': ['There is a significant participation gap, as 2 out of 3 students have 0 attempts and are not assessed.',
  'Performance improved steadily from Day 1 (40% accuracy) to Day 6 (80% accuracy).',
  "While most topics are strong, there is a specific failure point in one of the 'Variables' topic assessments."],
 'student_observations': ["Demo Student is the only active participant with 6 attempts and a 'medium' status.",
  'Demo Student 2 and Demo Student 3 are not_assessed.'],
 'weak_modules': [{'module': 'Python Basics',
   'accuracy': 66.67,
   'observation': "The mo

In [72]:
# Display the Faculty Summary
print("=" * 70)
print("FACULTY COURSE SUMMARY")
print("=" * 70)

print("\nCOURSE SUMMARY:")
print(overall_ai_result["course_summary"])

print("\nOVERALL PERFORMANCE:")
print(overall_ai_result["overall_performance"])


print("\nKEY FINDINGS:")
for finding in overall_ai_result["key_findings"]:
    print("•", finding)


print("\nSTUDENT OBSERVATIONS:")
for observation in overall_ai_result["student_observations"]:
    print("•", observation)


print("\nWEAK MODULES:")
for module in overall_ai_result["weak_modules"]:
    print(
        f"• {module['module']} "
        f"({module['accuracy']}%)"
    )
    print(
        f"  {module['observation']}"
    )


print("\nWEAK TOPICS:")
for topic in overall_ai_result["weak_topics"]:
    print(
        f"• {topic['topic']} "
        f"({topic['accuracy']}%)"
    )
    print(
        f"  {topic['observation']}"
    )


print("\nTEACHING RECOMMENDATIONS:")
for recommendation in overall_ai_result["teaching_recommendations"]:
    print("•", recommendation)


print("\nREVISION PRIORITIES:")
for priority in overall_ai_result["revision_priorities"]:
    print("•", priority)


print("\nFACULTY ACTION PLAN:")
for action in overall_ai_result["faculty_action_plan"]:
    print("•", action)

FACULTY COURSE SUMMARY

COURSE SUMMARY:
The course currently has 3 students enrolled in the Aarohan-2026 batch, with only 1 student actively participating in assessments for the 'Python Basics' module.

OVERALL PERFORMANCE:
The overall accuracy is 66.67% with an average score of 63.33%. Performance shows a positive upward trend over the 6-day period, starting from a minimum score of 40.00 and reaching a maximum of 80.00.

KEY FINDINGS:
• There is a significant participation gap, as 2 out of 3 students have 0 attempts and are not assessed.
• Performance improved steadily from Day 1 (40% accuracy) to Day 6 (80% accuracy).
• While most topics are strong, there is a specific failure point in one of the 'Variables' topic assessments.

STUDENT OBSERVATIONS:
• Demo Student is the only active participant with 6 attempts and a 'medium' status.
• Demo Student 2 and Demo Student 3 are not_assessed.

WEAK MODULES:
• Python Basics (66.67%)
  The module is flagged as 'Needs Improvement' due to an av

In [73]:
# Stage 4 — Save Overall AI Analysis to MySQL
# Check your existing tables
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

tables

['ai_code_labs',
 'ai_quiz_analysis',
 'days',
 'lessons',
 'modules',
 'question_attempts',
 'questions',
 'quiz_attempts',
 'quizzes',
 'students',
 'topics']

In [74]:
# ai_course_analysis
create_course_analysis_table = text("""
CREATE TABLE IF NOT EXISTS ai_course_analysis (
    id INT AUTO_INCREMENT PRIMARY KEY,

    batch VARCHAR(100),

    total_students INT,
    assessed_students INT,
    total_attempts INT,

    average_score FLOAT,
    overall_accuracy FLOAT,

    weak_modules JSON,
    weak_days JSON,
    weak_topics JSON,

    course_summary TEXT,
    overall_performance TEXT,

    key_findings JSON,
    student_observations JSON,

    teaching_recommendations JSON,
    revision_priorities JSON,
    faculty_action_plan JSON,

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

with engine.begin() as conn:
    conn.execute(create_course_analysis_table)

print("ai_course_analysis table created successfully.")

ai_course_analysis table created successfully.


In [75]:
# Create the save function
def save_overall_course_analysis(
    overall_result,
    overall_ai_result,
    batch="Aarohan-2026"
):

    course_overall = overall_result["course_overall"]

    student_df = overall_result["student_performance"]

    # Count students who actually attempted quizzes
    assessed_students = student_df[
        student_df["attempts"] > 0
    ].shape[0]

    # Convert weak areas into JSON-safe lists
    weak_modules = dataframe_to_records(
        overall_result["weak_modules"]
    )

    weak_days = dataframe_to_records(
        overall_result["weak_days"]
    )

    weak_topics = dataframe_to_records(
        overall_result["weak_topics"]
    )

    query = text("""
        INSERT INTO ai_course_analysis (

            batch,

            total_students,
            assessed_students,
            total_attempts,

            average_score,
            overall_accuracy,

            weak_modules,
            weak_days,
            weak_topics,

            course_summary,
            overall_performance,

            key_findings,
            student_observations,

            teaching_recommendations,
            revision_priorities,
            faculty_action_plan

        )

        VALUES (

            :batch,

            :total_students,
            :assessed_students,
            :total_attempts,

            :average_score,
            :overall_accuracy,

            :weak_modules,
            :weak_days,
            :weak_topics,

            :course_summary,
            :overall_performance,

            :key_findings,
            :student_observations,

            :teaching_recommendations,
            :revision_priorities,
            :faculty_action_plan
        )
    """)

    params = {
        "batch": batch,

        "total_students": course_overall["total_students"],
        "assessed_students": assessed_students,
        "total_attempts": course_overall["total_attempts"],

        "average_score": course_overall["average_score"],
        "overall_accuracy": course_overall["overall_accuracy"],

        "weak_modules": json.dumps(
            weak_modules,
            default=str
        ),

        "weak_days": json.dumps(
            weak_days,
            default=str
        ),

        "weak_topics": json.dumps(
            weak_topics,
            default=str
        ),

        "course_summary": overall_ai_result["course_summary"],

        "overall_performance":
            overall_ai_result["overall_performance"],

        "key_findings": json.dumps(
            overall_ai_result["key_findings"],
            default=str
        ),

        "student_observations": json.dumps(
            overall_ai_result["student_observations"],
            default=str
        ),

        "teaching_recommendations": json.dumps(
            overall_ai_result["teaching_recommendations"],
            default=str
        ),

        "revision_priorities": json.dumps(
            overall_ai_result["revision_priorities"],
            default=str
        ),

        "faculty_action_plan": json.dumps(
            overall_ai_result["faculty_action_plan"],
            default=str
        )
    }

    with engine.begin() as conn:
        result = conn.execute(query, params)

    return result.lastrowid

In [76]:
# Save the current analysis
analysis_id = save_overall_course_analysis(
    overall_result,
    overall_ai_result
)

print("Saved Analysis ID:", analysis_id)

Saved Analysis ID: 1


In [77]:
# Verify it in MySQL
with engine.connect() as conn:

    result = conn.execute(
        text("""
            SELECT
                id,
                batch,
                total_students,
                assessed_students,
                total_attempts,
                average_score,
                overall_accuracy,
                created_at
            FROM ai_course_analysis
            ORDER BY id DESC
        """)
    )

    saved_df = pd.DataFrame(
        result.fetchall(),
        columns=result.keys()
    )

display(saved_df)

,id,batch,total_students,assessed_students,total_attempts,average_score,overall_accuracy,created_at
0,1,Aarohan-2026,1,1,6,63.3333,66.67,2026-09-23 16:06:22


In [78]:
# Stage 5 — Create the Retrieval Layer
def get_latest_course_analysis(batch="Aarohan-2026"):

    query = text("""
        SELECT *
        FROM ai_course_analysis
        WHERE batch = :batch
        ORDER BY created_at DESC
        LIMIT 1
    """)

    with engine.connect() as conn:
        row = conn.execute(
            query,
            {"batch": batch}
        ).mappings().first()

    if row is None:
        return None

    return dict(row)

In [79]:
latest_analysis = get_latest_course_analysis()

latest_analysis

{'id': 1,
 'batch': 'Aarohan-2026',
 'total_students': 1,
 'assessed_students': 1,
 'total_attempts': 6,
 'average_score': 63.3333,
 'overall_accuracy': 66.67,
 'weak_modules': '[{"status": "Needs Improvement", "accuracy": "66.66666666666666666666666667", "attempts": 6, "students": 1, "module_id": 1, "module_name": "Python Basics", "average_score": "63.333333", "total_questions": "30", "correct_questions": "20"}]',
 'weak_days': '[{"day_id": 6, "status": "Needs Immediate Attention", "accuracy": "40.0", "attempts": 1, "students": 1, "module_id": 1, "day_number": 1, "module_name": "Python Basics", "average_score": "40.000000", "total_questions": "5", "correct_questions": "2"}, {"day_id": 5, "status": "Needs Improvement", "accuracy": "60.0", "attempts": 1, "students": 1, "module_id": 1, "day_number": 2, "module_name": "Python Basics", "average_score": "60.000000", "total_questions": "5", "correct_questions": "3"}, {"day_id": 4, "status": "Needs Improvement", "accuracy": "60.0", "attempts"

In [80]:
# Stage 8 — Complete Personalized Learning Loop.
# ai_result
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE ai_quiz_analysis
        ADD COLUMN ai_result JSON NULL
    """))

print("ai_result column added successfully.")

ai_result column added successfully.


In [87]:
type(result)

dict

In [88]:
result.keys()

dict_keys(['student', 'overall_score', 'risk_level', 'quiz_attempts', 'question_analysis', 'wrong_questions', 'topic_performance', 'weak_topics', 'primary_topic', 'ai_result'])

In [89]:
print(
    json.dumps(
        result["ai_result"],
        indent=2,
        default=str
    )
)

{
  "student": "Demo Student",
  "risk_level": "medium",
  "weak_topic": "Variables",
  "learning_gap": "The student is struggling with the fundamental concept of variables, specifically how they store and retrieve values, as evidenced by multiple incorrect answers across different quizzes.",
  "simple_explanation": "A variable is like a labeled container in a computer's memory. You can put a piece of information inside it, give the container a name, and then use that name later to find the information you stored.",
  "five_year_old_explanation": "Imagine you have a bunch of toy boxes. You put your favorite dinosaur in one box and write 'My Toy' on the outside. Whenever you want your dinosaur, you don't have to search the whole room; you just look for the box that says 'My Toy'!",
  "real_world_analogy": "Think of a variable like a contact name in your phone. The name 'Mom' is the variable, and her actual phone number is the value stored inside. You don't need to remember the number; y

In [91]:
def save_complete_personalized_analysis(result):

    student = result["student"]
    ai_result = result["ai_result"]

    quiz_attempts = result["quiz_attempts"]

    # ---------------------------------------------------------
    # CHECK QUIZ ATTEMPTS
    # ---------------------------------------------------------

    if quiz_attempts is None:
        print("No quiz attempts found.")
        return None

    if isinstance(quiz_attempts, pd.DataFrame):

        if quiz_attempts.empty:
            print("No quiz attempts found.")
            return None

        # Latest attempt
        latest_attempt = quiz_attempts.iloc[-1].to_dict()

    elif isinstance(quiz_attempts, list):

        if len(quiz_attempts) == 0:
            print("No quiz attempts found.")
            return None

        latest_attempt = quiz_attempts[-1]

        if hasattr(latest_attempt, "_mapping"):
            latest_attempt = dict(
                latest_attempt._mapping
            )

    else:
        print(
            "Unsupported quiz_attempts type:",
            type(quiz_attempts)
        )
        return None

    # ---------------------------------------------------------
    # GET ATTEMPT INFORMATION
    # ---------------------------------------------------------

    attempt_id = latest_attempt["attempt_id"]
    quiz_id = latest_attempt["quiz_id"]

    score = latest_attempt["score"]

    total_questions = latest_attempt[
        "total_questions"
    ]

    correct_questions = latest_attempt[
        "correct_questions"
    ]

    wrong_questions = (
        total_questions - correct_questions
    )

    # ---------------------------------------------------------
    # WEAK TOPICS
    # ---------------------------------------------------------

    weak_topics = result["weak_topics"]

    if isinstance(weak_topics, pd.DataFrame):
        weak_topics_json = weak_topics.to_dict(
            orient="records"
        )
    else:
        weak_topics_json = weak_topics

    # ---------------------------------------------------------
    # QUESTION ANALYSIS
    # ---------------------------------------------------------

    question_analysis = result["question_analysis"]

    if isinstance(question_analysis, pd.DataFrame):
        question_analysis_json = (
            question_analysis.to_dict(
                orient="records"
            )
        )
    else:
        question_analysis_json = question_analysis

    # ---------------------------------------------------------
    # INSERT INTO DATABASE
    # ---------------------------------------------------------

    query = text("""
        INSERT INTO ai_quiz_analysis (

            student_id,
            attempt_id,
            quiz_id,

            score,
            total_questions,
            correct_questions,
            wrong_questions,

            weak_topics,
            question_analysis,

            ai_summary,
            recommendations,

            ai_result

        )

        VALUES (

            :student_id,
            :attempt_id,
            :quiz_id,

            :score,
            :total_questions,
            :correct_questions,
            :wrong_questions,

            :weak_topics,
            :question_analysis,

            :ai_summary,
            :recommendations,

            :ai_result

        )
    """)

    params = {

        "student_id": student["id"],

        "attempt_id": attempt_id,

        "quiz_id": quiz_id,

        "score": score,

        "total_questions": total_questions,

        "correct_questions": correct_questions,

        "wrong_questions": wrong_questions,

        "weak_topics": json.dumps(
            weak_topics_json,
            default=str
        ),

        "question_analysis": json.dumps(
            question_analysis_json,
            default=str
        ),

        "ai_summary": ai_result.get(
            "learning_gap",
            ""
        ),

        "recommendations": json.dumps(
            {
                "recommended_lesson":
                    ai_result.get(
                        "recommended_lesson"
                    ),

                "practice_activity":
                    ai_result.get(
                        "practice_activity"
                    ),

                "motivation":
                    ai_result.get(
                        "motivation"
                    )
            },
            default=str
        ),

        "ai_result": json.dumps(
            ai_result,
            default=str
        )
    }

    # ---------------------------------------------------------
    # SAVE
    # ---------------------------------------------------------

    with engine.begin() as conn:

        db_result = conn.execute(
            query,
            params
        )

    print(
        "Personalized AI saved successfully."
    )

    return db_result.lastrowid

In [92]:
result = run_personalized_ai(1)

STARTING PERSONALIZED AI ANALYSIS

👤 Student: Demo Student
📊 Quiz Attempts: 6
📈 Overall Score: 66.67%
⚠️ Risk Level: medium
📝 Questions Analyzed: 30
❌ Wrong Questions: 4
🎯 Weak Topics Found: 1
📚 Primary Weak Topic: Variables

🤖 Generating personalized recommendation...

✅ Personalized AI analysis completed!


In [94]:
print(type(result))
print(type(result["quiz_attempts"]))

<class 'dict'>
<class 'pandas.core.frame.DataFrame'>


In [98]:
from sqlalchemy import text

with engine.connect() as conn:

    print("=== QUIZ ATTEMPTS FOR STUDENT 1 ===")

    rows = conn.execute(
        text("""
            SELECT
                id,
                student_id,
                quiz_id,
                score,
                total_questions,
                correct_questions
            FROM quiz_attempts
            WHERE student_id = 1
            ORDER BY id
        """)
    ).mappings().all()

    for row in rows:
        print(dict(row))


    print("\n=== EXISTING AI ANALYSIS ===")

    rows = conn.execute(
        text("""
            SELECT
                id,
                student_id,
                attempt_id,
                quiz_id,
                score
            FROM ai_quiz_analysis
            WHERE student_id = 1
            ORDER BY id
        """)
    ).mappings().all()

    for row in rows:
        print(dict(row))


    print("\n=== FOREIGN KEYS ===")

    rows = conn.execute(
        text("""
            SELECT
                CONSTRAINT_NAME,
                COLUMN_NAME,
                REFERENCED_TABLE_NAME,
                REFERENCED_COLUMN_NAME
            FROM information_schema.KEY_COLUMN_USAGE
            WHERE TABLE_SCHEMA = DATABASE()
              AND TABLE_NAME = 'ai_quiz_analysis'
              AND REFERENCED_TABLE_NAME IS NOT NULL
        """)
    ).mappings().all()

    for row in rows:
        print(dict(row))

=== QUIZ ATTEMPTS FOR STUDENT 1 ===
{'id': 1, 'student_id': 1, 'quiz_id': 1, 'score': Decimal('80.00'), 'total_questions': 5, 'correct_questions': 4}
{'id': 2, 'student_id': 1, 'quiz_id': 2, 'score': Decimal('75.00'), 'total_questions': 5, 'correct_questions': 4}
{'id': 3, 'student_id': 1, 'quiz_id': 3, 'score': Decimal('70.00'), 'total_questions': 5, 'correct_questions': 4}
{'id': 4, 'student_id': 1, 'quiz_id': 4, 'score': Decimal('55.00'), 'total_questions': 5, 'correct_questions': 3}
{'id': 5, 'student_id': 1, 'quiz_id': 5, 'score': Decimal('60.00'), 'total_questions': 5, 'correct_questions': 3}
{'id': 6, 'student_id': 1, 'quiz_id': 6, 'score': Decimal('40.00'), 'total_questions': 5, 'correct_questions': 2}

=== EXISTING AI ANALYSIS ===
{'id': 1, 'student_id': 1, 'attempt_id': 1, 'quiz_id': 1, 'score': Decimal('66.67')}

=== FOREIGN KEYS ===


In [96]:
print("quiz_attempts:")
print(result["quiz_attempts"].to_string())

print("\nquestion_analysis attempt IDs:")

print(
    result["question_analysis"]["attempt_id"]
    .dropna()
    .unique()
)

quiz_attempts:
   attempt_id        completed_at  correct_questions  day_id  day_number  module_id    module_name  quiz_id  quiz_title  score  student_id  total_questions
0           6 2026-09-01 10:00:00                  2       6           1          1  Python Basics        6  Day 1 Quiz  40.00           1                5
1           5 2026-09-02 10:00:00                  3       5           2          1  Python Basics        5  Day 2 Quiz  60.00           1                5
2           4 2026-09-03 10:00:00                  3       4           3          1  Python Basics        4  Day 3 Quiz  55.00           1                5
3           3 2026-09-04 10:00:00                  4       3           4          1  Python Basics        3  Day 4 Quiz  70.00           1                5
4           2 2026-09-05 10:00:00                  4       2           5          1  Python Basics        2  Day 5 Quiz  75.00           1                5
5           1 2026-09-06 10:00:00                

In [99]:
from sqlalchemy import text

with engine.connect() as conn:

    print("===== QUIZ ATTEMPTS =====")

    rows = conn.execute(
        text("""
            SELECT
                id,
                student_id,
                quiz_id,
                score,
                total_questions,
                correct_questions
            FROM quiz_attempts
            WHERE student_id = 1
            ORDER BY id
        """)
    ).mappings().all()

    for row in rows:
        print(dict(row))

    print("\n===== AI ANALYSIS =====")

    rows = conn.execute(
        text("""
            SELECT
                id,
                student_id,
                attempt_id,
                quiz_id,
                score
            FROM ai_quiz_analysis
            WHERE student_id = 1
            ORDER BY id
        """)
    ).mappings().all()

    for row in rows:
        print(dict(row))

    print("\n===== FOREIGN KEYS =====")

    rows = conn.execute(
        text("""
            SELECT
                CONSTRAINT_NAME,
                COLUMN_NAME,
                REFERENCED_TABLE_NAME,
                REFERENCED_COLUMN_NAME
            FROM information_schema.KEY_COLUMN_USAGE
            WHERE TABLE_SCHEMA = DATABASE()
              AND TABLE_NAME = 'ai_quiz_analysis'
              AND REFERENCED_TABLE_NAME IS NOT NULL
        """)
    ).mappings().all()

    for row in rows:
        print(dict(row))

===== QUIZ ATTEMPTS =====
{'id': 1, 'student_id': 1, 'quiz_id': 1, 'score': Decimal('80.00'), 'total_questions': 5, 'correct_questions': 4}
{'id': 2, 'student_id': 1, 'quiz_id': 2, 'score': Decimal('75.00'), 'total_questions': 5, 'correct_questions': 4}
{'id': 3, 'student_id': 1, 'quiz_id': 3, 'score': Decimal('70.00'), 'total_questions': 5, 'correct_questions': 4}
{'id': 4, 'student_id': 1, 'quiz_id': 4, 'score': Decimal('55.00'), 'total_questions': 5, 'correct_questions': 3}
{'id': 5, 'student_id': 1, 'quiz_id': 5, 'score': Decimal('60.00'), 'total_questions': 5, 'correct_questions': 3}
{'id': 6, 'student_id': 1, 'quiz_id': 6, 'score': Decimal('40.00'), 'total_questions': 5, 'correct_questions': 2}

===== AI ANALYSIS =====
{'id': 1, 'student_id': 1, 'attempt_id': 1, 'quiz_id': 1, 'score': Decimal('66.67')}

===== FOREIGN KEYS =====


In [100]:
print("AI result type:", type(result["ai_result"]))
print("Quiz attempts type:", type(result["quiz_attempts"]))
print(
    "Question analysis type:",
    type(result["question_analysis"])
)

AI result type: <class 'dict'>
Quiz attempts type: <class 'pandas.core.frame.DataFrame'>
Question analysis type: <class 'pandas.core.frame.DataFrame'>


In [101]:
# Create practice quiz table
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS ai_practice_attempts (
            id INT AUTO_INCREMENT PRIMARY KEY,

            student_id INT NOT NULL,

            original_attempt_id INT NULL,

            topic_id INT NULL,

            score FLOAT,

            total_questions INT,

            correct_questions INT,

            wrong_questions INT,

            answers JSON,

            completed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

            FOREIGN KEY (student_id)
                REFERENCES students(id)
        )
    """))

print("ai_practice_attempts table created successfully.")

ai_practice_attempts table created successfully.


In [102]:
# Verify the table
with engine.connect() as conn:
    rows = conn.execute(
        text("SHOW TABLES")
    ).fetchall()

    for row in rows:
        print(row[0])

ai_code_labs
ai_course_analysis
ai_practice_attempts
ai_quiz_analysis
days
lessons
modules
question_attempts
questions
quiz_attempts
quizzes
students
topics


In [103]:
# Check the attempts used by the personalized analysis

print("Quiz Attempts:")
print(result["quiz_attempts"].to_string(index=False))

print("\nQuestion Analysis Attempt IDs:")
print(result["question_analysis"]["attempt_id"].unique())

Quiz Attempts:
 attempt_id        completed_at  correct_questions  day_id  day_number  module_id   module_name  quiz_id quiz_title score  student_id  total_questions
          6 2026-09-01 10:00:00                  2       6           1          1 Python Basics        6 Day 1 Quiz 40.00           1                5
          5 2026-09-02 10:00:00                  3       5           2          1 Python Basics        5 Day 2 Quiz 60.00           1                5
          4 2026-09-03 10:00:00                  3       4           3          1 Python Basics        4 Day 3 Quiz 55.00           1                5
          3 2026-09-04 10:00:00                  4       3           4          1 Python Basics        3 Day 4 Quiz 70.00           1                5
          2 2026-09-05 10:00:00                  4       2           5          1 Python Basics        2 Day 5 Quiz 75.00           1                5
          1 2026-09-06 10:00:00                  4       1           6         

In [105]:
from sqlalchemy import text

try:
    with engine.connect() as conn:
        rows = conn.execute(text("""
            SELECT
                id,
                student_id,
                quiz_id,
                score,
                total_questions,
                correct_questions,
                wrong_questions,
                completed_at
            FROM quiz_attempts
            WHERE student_id = :student_id
            ORDER BY id
        """), {"student_id": 1}).fetchall()

    print("Query successful!")
    for row in rows:
        print(row)

except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR MESSAGE:")
    print(str(e))

ERROR TYPE: OperationalError
ERROR MESSAGE:
(pymysql.err.OperationalError) (1054, "Unknown column 'wrong_questions' in 'field list'")
[SQL: 
            SELECT
                id,
                student_id,
                quiz_id,
                score,
                total_questions,
                correct_questions,
                wrong_questions,
                completed_at
            FROM quiz_attempts
            WHERE student_id = %(student_id)s
            ORDER BY id
        ]
[parameters: {'student_id': 1}]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [107]:
# Check your actual quiz_attempts columns
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(
        text("SHOW COLUMNS FROM quiz_attempts")
    ).fetchall()

for row in rows:
    print(row)

('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('student_id', 'int', 'NO', 'MUL', None, '')
('quiz_id', 'int', 'NO', 'MUL', None, '')
('score', 'decimal(5,2)', 'NO', '', None, '')
('total_questions', 'int', 'NO', '', None, '')
('correct_questions', 'int', 'NO', '', None, '')
('completed_at', 'datetime', 'NO', '', None, '')


In [108]:
# Check the actual attempts
with engine.connect() as conn:
    rows = conn.execute(
        text("""
            SELECT *
            FROM quiz_attempts
            WHERE student_id = :student_id
            ORDER BY id
        """),
        {"student_id": 1}
    ).fetchall()

for row in rows:
    print(row)

(1, 1, 1, Decimal('80.00'), 5, 4, datetime.datetime(2026, 9, 6, 10, 0))
(2, 1, 2, Decimal('75.00'), 5, 4, datetime.datetime(2026, 9, 5, 10, 0))
(3, 1, 3, Decimal('70.00'), 5, 4, datetime.datetime(2026, 9, 4, 10, 0))
(4, 1, 4, Decimal('55.00'), 5, 3, datetime.datetime(2026, 9, 3, 10, 0))
(5, 1, 5, Decimal('60.00'), 5, 3, datetime.datetime(2026, 9, 2, 10, 0))
(6, 1, 6, Decimal('40.00'), 5, 2, datetime.datetime(2026, 9, 1, 10, 0))


In [111]:
from sqlalchemy import text
import json
import pandas as pd


# ============================================================
# 1. RUN PERSONALIZED AI ANALYSIS
# ============================================================

student_id = 1

result = run_personalized_ai(student_id)

print("Personalized AI analysis generated successfully.")
print("Result type:", type(result))
print("Result keys:", result.keys())


# ============================================================
# 2. GET QUESTION ANALYSIS
# ============================================================

question_analysis = result.get("question_analysis")

if question_analysis is None or question_analysis.empty:
    raise ValueError("No question analysis found.")

print("\nQuestion Analysis:")
print(question_analysis.to_string(index=False))


# ============================================================
# 3. GET ACTUAL ATTEMPT ID
# ============================================================

attempt_ids = question_analysis["attempt_id"].dropna().unique()

print("\nAttempt IDs found:", attempt_ids)

if len(attempt_ids) == 0:
    raise ValueError("No attempt_id found in question analysis.")

# Use the latest/highest analyzed attempt
attempt_id = int(max(attempt_ids))

print("Using attempt_id:", attempt_id)


# ============================================================
# 4. VERIFY THAT ATTEMPT EXISTS IN DATABASE
# ============================================================

with engine.connect() as db_conn:

    db_attempt = db_conn.execute(
        text("""
            SELECT
                id,
                student_id,
                quiz_id,
                score,
                total_questions,
                correct_questions
            FROM quiz_attempts
            WHERE id = :attempt_id
              AND student_id = :student_id
        """),
        {
            "attempt_id": attempt_id,
            "student_id": student_id
        }
    ).mappings().first()


if not db_attempt:
    raise ValueError(
        f"Attempt {attempt_id} does not exist for student {student_id}."
    )

print("\nDatabase Attempt:")
print(dict(db_attempt))


# ============================================================
# 5. PREPARE AI RESULT
# ============================================================

ai_result = result.get("ai_result")

if not ai_result:
    raise ValueError("AI result is empty.")

print("\nAI Result:")
print(json.dumps(ai_result, indent=2, ensure_ascii=False))


# ============================================================
# 6. PREPARE WEAK TOPICS
# ============================================================

weak_topics_df = result.get("weak_topics")

if isinstance(weak_topics_df, pd.DataFrame):

    weak_topics = weak_topics_df.to_dict(orient="records")

elif weak_topics_df:

    weak_topics = weak_topics_df

else:

    weak_topics = []


# ============================================================
# 7. PREPARE QUESTION ANALYSIS JSON
# ============================================================

question_analysis_json = question_analysis.to_dict(
    orient="records"
)


# ============================================================
# 8. PREPARE AI JSON
# ============================================================

ai_result_json = json.dumps(
    ai_result,
    ensure_ascii=False
)


# ============================================================
# 9. SAVE PERSONALIZED ANALYSIS
# ============================================================

save_query = text("""
    INSERT INTO ai_quiz_analysis (
        student_id,
        attempt_id,
        quiz_id,
        score,
        total_questions,
        correct_questions,
        wrong_questions,
        weak_topics,
        question_analysis,
        ai_summary,
        ai_result
    )
    VALUES (
        :student_id,
        :attempt_id,
        :quiz_id,
        :score,
        :total_questions,
        :correct_questions,
        :wrong_questions,
        :weak_topics,
        :question_analysis,
        :ai_summary,
        :ai_result
    )
""")


with engine.begin() as db_conn:

    db_conn.execute(
        save_query,
        {
            "student_id": student_id,

            # IMPORTANT:
            # This is the SAME attempt used in question_analysis
            "attempt_id": int(db_attempt["id"]),

            "quiz_id": int(db_attempt["quiz_id"]),

            "score": float(db_attempt["score"]),

            "total_questions": int(
                db_attempt["total_questions"]
            ),

            "correct_questions": int(
                db_attempt["correct_questions"]
            ),

            "wrong_questions": (
                int(db_attempt["total_questions"])
                - int(db_attempt["correct_questions"])
            ),

            "weak_topics": json.dumps(
                weak_topics,
                ensure_ascii=False
            ),

            "question_analysis": json.dumps(
                question_analysis_json,
                ensure_ascii=False
            ),

            "ai_summary": ai_result.get(
                "learning_gap",
                ai_result.get(
                    "simple_explanation",
                    ""
                )
            ),

            "ai_result": ai_result_json
        }
    )


print("\n========================================")
print("PERSONALIZED ANALYSIS SAVED SUCCESSFULLY")
print("========================================")
print("Student ID :", student_id)
print("Attempt ID :", attempt_id)
print("Quiz ID    :", db_attempt["quiz_id"])
print("Score      :", db_attempt["score"])
print("Weak Topic :", ai_result.get("weak_topic"))

STARTING PERSONALIZED AI ANALYSIS

👤 Student: Demo Student
📊 Quiz Attempts: 6
📈 Overall Score: 66.67%
⚠️ Risk Level: medium
📝 Questions Analyzed: 30
❌ Wrong Questions: 4
🎯 Weak Topics Found: 1
📚 Primary Weak Topic: Variables

🤖 Generating personalized recommendation...

✅ Personalized AI analysis completed!
Personalized AI analysis generated successfully.
Result type: <class 'dict'>
Result keys: dict_keys(['student', 'overall_score', 'risk_level', 'quiz_attempts', 'question_analysis', 'wrong_questions', 'topic_performance', 'weak_topics', 'primary_topic', 'ai_result'])

Question Analysis:
 attempt_id correct_answer  day_id  day_number  is_correct  lesson_id             lesson_title  module_id   module_name  question_attempt_id  question_id             question_text quiz_title student_answer      topic  topic_id
          6              C       6           1           0          6 Lesson 1 - Core Concepts          1 Python Basics                   26           26 Question 5 for Day 1 Qu

In [112]:
with engine.connect() as db_conn:

    saved = db_conn.execute(
        text("""
            SELECT
                id,
                student_id,
                attempt_id,
                quiz_id,
                score,
                total_questions,
                correct_questions,
                wrong_questions,
                ai_summary,
                created_at
            FROM ai_quiz_analysis
            WHERE student_id = :student_id
            ORDER BY id DESC
            LIMIT 1
        """),
        {"student_id": 1}
    ).mappings().first()

print(dict(saved) if saved else "No record found")

{'id': 3, 'student_id': 1, 'attempt_id': 6, 'quiz_id': 6, 'score': Decimal('40.00'), 'total_questions': 5, 'correct_questions': 2, 'wrong_questions': 3, 'ai_summary': 'The student is struggling with the fundamental concept of variables, specifically how they store and retrieve information, as evidenced by multiple incorrect answers across different quizzes.', 'created_at': datetime.datetime(2026, 9, 26, 11, 29, 48)}


In [113]:
# AI Practice & Re-learning
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS ai_practice_attempts (
            id INT AUTO_INCREMENT PRIMARY KEY,
            student_id INT NOT NULL,
            original_attempt_id INT NOT NULL,
            topic_id INT NULL,

            score FLOAT NOT NULL,
            total_questions INT NOT NULL,
            correct_questions INT NOT NULL,
            wrong_questions INT NOT NULL,

            answers JSON NULL,

            completed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

            FOREIGN KEY (student_id)
                REFERENCES students(id)
        )
    """))

print("ai_practice_attempts table created successfully.")

ai_practice_attempts table created successfully.


In [114]:
with engine.connect() as conn:
    rows = conn.execute(
        text("SHOW COLUMNS FROM ai_practice_attempts")
    ).fetchall()

for row in rows:
    print(row)

('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('student_id', 'int', 'NO', 'MUL', None, '')
('original_attempt_id', 'int', 'YES', '', None, '')
('topic_id', 'int', 'YES', '', None, '')
('score', 'float', 'YES', '', None, '')
('total_questions', 'int', 'YES', '', None, '')
('correct_questions', 'int', 'YES', '', None, '')
('wrong_questions', 'int', 'YES', '', None, '')
('answers', 'json', 'YES', '', None, '')
('completed_at', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')


In [115]:
import json
import re
from sqlalchemy import text


def get_latest_personalized_analysis(student_id):
    """
    Get the latest saved personalized AI analysis.
    """

    with engine.connect() as conn:
        row = conn.execute(
            text("""
                SELECT
                    id,
                    student_id,
                    attempt_id,
                    quiz_id,
                    score,
                    total_questions,
                    correct_questions,
                    wrong_questions,
                    weak_topics,
                    question_analysis,
                    ai_summary,
                    ai_result,
                    created_at
                FROM ai_quiz_analysis
                WHERE student_id = :student_id
                ORDER BY id DESC
                LIMIT 1
            """),
            {"student_id": student_id}
        ).mappings().first()

    if not row:
        return None

    result = dict(row)

    # Convert JSON fields
    for field in [
        "weak_topics",
        "question_analysis",
        "ai_result"
    ]:
        value = result.get(field)

        if isinstance(value, str):
            try:
                result[field] = json.loads(value)
            except json.JSONDecodeError:
                result[field] = {}

    return result


print("Function created successfully.")

Function created successfully.


In [116]:
# Create the practice-question prompt
def build_practice_quiz_prompt(analysis, number_of_questions=5):

    ai_result = analysis.get("ai_result", {})

    weak_topic = ai_result.get(
        "weak_topic",
        "Variables"
    )

    simple_explanation = ai_result.get(
        "simple_explanation",
        ""
    )

    five_year_old_explanation = ai_result.get(
        "five_year_old_explanation",
        ""
    )

    recommended_lesson = ai_result.get(
        "recommended_lesson",
        ""
    )

    wrong_questions = analysis.get(
        "question_analysis",
        []
    )

    # Keep only incorrect questions
    wrong_only = []

    if isinstance(wrong_questions, list):

        for q in wrong_questions:

            if isinstance(q, dict):

                is_correct = q.get("is_correct")

                if is_correct in [False, 0, "false", "False"]:

                    wrong_only.append(q)

    prompt = f"""
You are an AI learning assistant for an LMS.

Create a practice quiz for a student who needs
additional learning support.

STUDENT WEAK TOPIC:
{weak_topic}

AI LEARNING GAP:
{simple_explanation}

SIMPLE EXPLANATION:
{five_year_old_explanation}

RECOMMENDED LESSON:
{recommended_lesson}

ORIGINAL INCORRECT QUESTIONS:
{json.dumps(wrong_only, indent=2, ensure_ascii=False)}

Requirements:

1. Generate exactly {number_of_questions} NEW questions.
2. Questions must focus mainly on the weak topic:
   {weak_topic}
3. Do NOT copy the original questions.
4. Use different examples and values.
5. Start from easy questions and gradually increase difficulty.
6. Questions must test understanding, not memorization.
7. Include practical examples where appropriate.
8. Return ONLY valid JSON.
9. Do not use Markdown code fences.

Return exactly this structure:

{{
    "topic": "{weak_topic}",
    "difficulty": "beginner",
    "questions": [
        {{
            "id": 1,
            "question": "...",
            "options": [
                "A",
                "B",
                "C",
                "D"
            ],
            "correct_answer": "A",
            "explanation": "..."
        }}
    ]
}}
"""

    return prompt


print("Practice prompt function created.")

Practice prompt function created.


In [117]:
# Generate the practice quiz
def generate_practice_quiz(student_id, number_of_questions=5):

    analysis = get_latest_personalized_analysis(student_id)

    if not analysis:
        raise ValueError(
            f"No personalized analysis found for student {student_id}"
        )

    prompt = build_practice_quiz_prompt(
        analysis,
        number_of_questions
    )

    response = llm.generate(prompt)

    ai_text = extract_ai_text(response)

    # Remove Markdown JSON fences if Ollama adds them
    ai_text = ai_text.strip()

    ai_text = re.sub(
        r"^```json\s*",
        "",
        ai_text,
        flags=re.IGNORECASE
    )

    ai_text = re.sub(
        r"\s*```$",
        "",
        ai_text
    )

    practice_quiz = json.loads(ai_text)

    return {
        "student_id": student_id,
        "original_attempt_id": analysis["attempt_id"],
        "topic": practice_quiz.get("topic"),
        "difficulty": practice_quiz.get("difficulty"),
        "questions": practice_quiz.get("questions", [])
    }

In [119]:
# Run it for your student
practice_quiz = generate_practice_quiz(
    student_id=1,
    number_of_questions=5
)

print(json.dumps(
    practice_quiz,
    indent=2,
    ensure_ascii=False
))

{
  "student_id": 1,
  "original_attempt_id": 6,
  "topic": "Variables",
  "difficulty": "beginner",
  "questions": [
    {
      "id": 1,
      "question": "Using the 'toy box' analogy, what does the label on the outside of the box represent in programming?",
      "options": [
        "The data stored inside",
        "The variable name",
        "The computer's memory",
        "The program's output"
      ],
      "correct_answer": "B",
      "explanation": "The label on the box is like the variable name; it tells the computer which specific 'container' to look in to find your information."
    },
    {
      "id": 2,
      "question": "If you write 'score = 10' in your code, what is the value being stored in the variable 'score'?",
      "options": [
        "score",
        "=",
        "10",
        "Variable"
      ],
      "correct_answer": "C",
      "explanation": "In the statement 'score = 10', 'score' is the name of the container and '10' is the actual piece of information

In [120]:
# Create the submission function
def submit_practice_quiz(
    student_id,
    practice_quiz,
    answers
):
    """
    Evaluate the student's AI-generated practice quiz
    and save the practice attempt.
    
    answers example:
    {
        1: "B",
        2: "C",
        3: "A",
        4: "D",
        5: "B"
    }
    """

    questions = practice_quiz.get("questions", [])

    if not questions:
        raise ValueError("No practice questions found.")

    correct_questions = 0
    answer_records = []

    for question in questions:

        question_id = question.get("id")
        correct_answer = question.get("correct_answer")

        student_answer = answers.get(question_id)

        is_correct = (
            student_answer is not None
            and student_answer.upper() == correct_answer.upper()
        )

        if is_correct:
            correct_questions += 1

        answer_records.append({
            "question_id": question_id,
            "student_answer": student_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct
        })

    total_questions = len(questions)

    wrong_questions = (
        total_questions - correct_questions
    )

    score = (
        correct_questions / total_questions
    ) * 100

    # Get topic ID
    topic_id = None

    with engine.connect() as conn:

        topic_row = conn.execute(
            text("""
                SELECT id
                FROM topics
                WHERE name = :topic
                LIMIT 1
            """),
            {
                "topic": practice_quiz.get("topic")
            }
        ).first()

        if topic_row:
            topic_id = topic_row[0]

    # Save practice attempt
    with engine.begin() as conn:

        conn.execute(
            text("""
                INSERT INTO ai_practice_attempts (
                    student_id,
                    original_attempt_id,
                    topic_id,
                    score,
                    total_questions,
                    correct_questions,
                    wrong_questions,
                    answers
                )
                VALUES (
                    :student_id,
                    :original_attempt_id,
                    :topic_id,
                    :score,
                    :total_questions,
                    :correct_questions,
                    :wrong_questions,
                    :answers
                )
            """),
            {
                "student_id": student_id,
                "original_attempt_id": practice_quiz[
                    "original_attempt_id"
                ],
                "topic_id": topic_id,
                "score": score,
                "total_questions": total_questions,
                "correct_questions": correct_questions,
                "wrong_questions": wrong_questions,
                "answers": json.dumps(
                    answer_records,
                    ensure_ascii=False
                )
            }
        )

    return {
        "student_id": student_id,
        "original_attempt_id": practice_quiz[
            "original_attempt_id"
        ],
        "topic": practice_quiz.get("topic"),
        "score": round(score, 2),
        "total_questions": total_questions,
        "correct_questions": correct_questions,
        "wrong_questions": wrong_questions,
        "answers": answer_records
    }

In [121]:
answers = {
    1: "B",
    2: "C",
    3: "A",
    4: "D",
    5: "B"
}

practice_result = submit_practice_quiz(
    student_id=1,
    practice_quiz=practice_quiz,
    answers=answers
)

print(json.dumps(
    practice_result,
    indent=2,
    ensure_ascii=False
))

{
  "student_id": 1,
  "original_attempt_id": 6,
  "topic": "Variables",
  "score": 60.0,
  "total_questions": 5,
  "correct_questions": 3,
  "wrong_questions": 2,
  "answers": [
    {
      "question_id": 1,
      "student_answer": "B",
      "correct_answer": "B",
      "is_correct": true
    },
    {
      "question_id": 2,
      "student_answer": "C",
      "correct_answer": "C",
      "is_correct": true
    },
    {
      "question_id": 3,
      "student_answer": "A",
      "correct_answer": "C",
      "is_correct": false
    },
    {
      "question_id": 4,
      "student_answer": "D",
      "correct_answer": "C",
      "is_correct": false
    },
    {
      "question_id": 5,
      "student_answer": "B",
      "correct_answer": "B",
      "is_correct": true
    }
  ]
}


In [123]:
# AI Improvement Analysis.
def analyze_practice_improvement(practice_result):
    """
    Compare original quiz performance with AI practice performance.
    """

    student_id = practice_result["student_id"]
    original_attempt_id = practice_result["original_attempt_id"]
    topic = practice_result["topic"]

    practice_score = practice_result["score"]

    # Get original quiz score
    with engine.connect() as conn:

        original = conn.execute(
            text("""
                SELECT
                    score,
                    total_questions,
                    correct_questions
                FROM quiz_attempts
                WHERE id = :attempt_id
                  AND student_id = :student_id
                LIMIT 1
            """),
            {
                "attempt_id": original_attempt_id,
                "student_id": student_id
            }
        ).mappings().first()

    if not original:
        raise ValueError(
            "Original quiz attempt not found."
        )

    original_score = float(original["score"])

    improvement = round(
        practice_score - original_score,
        2
    )

    # Determine learning status
    if practice_score >= 80:
        status = "mastered"
        recommendation = (
            f"The student has demonstrated good understanding "
            f"of {topic}. Move to the next topic."
        )

    elif practice_score >= 60:
        status = "improving"
        recommendation = (
            f"The student is improving in {topic}, "
            f"but needs some additional practice."
        )

    else:
        status = "needs_more_practice"
        recommendation = (
            f"The student still needs additional explanation "
            f"and practice on {topic}."
        )

    return {
        "student_id": student_id,
        "original_attempt_id": original_attempt_id,
        "topic": topic,
        "original_score": original_score,
        "practice_score": practice_score,
        "improvement": improvement,
        "status": status,
        "recommendation": recommendation
    }

In [124]:
improvement_result = analyze_practice_improvement(
    practice_result
)

print(
    json.dumps(
        improvement_result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "student_id": 1,
  "original_attempt_id": 6,
  "topic": "Variables",
  "original_score": 40.0,
  "practice_score": 60.0,
  "improvement": 20.0,
  "status": "improving",
  "recommendation": "The student is improving in Variables, but needs some additional practice."
}


In [125]:
with engine.begin() as conn:

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS ai_learning_progress (
            id INT AUTO_INCREMENT PRIMARY KEY,

            student_id INT NOT NULL,
            original_attempt_id INT NOT NULL,

            topic_id INT NULL,
            topic_name VARCHAR(255),

            original_score FLOAT,
            practice_score FLOAT,
            improvement FLOAT,

            status VARCHAR(50),
            recommendation TEXT,

            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

            FOREIGN KEY (student_id)
                REFERENCES students(id)
        )
    """))

print("ai_learning_progress table created successfully.")

ai_learning_progress table created successfully.


In [126]:
with engine.begin() as conn:

    conn.execute(
        text("""
            INSERT INTO ai_learning_progress (
                student_id,
                original_attempt_id,
                topic_id,
                topic_name,
                original_score,
                practice_score,
                improvement,
                status,
                recommendation
            )
            VALUES (
                :student_id,
                :original_attempt_id,
                :topic_id,
                :topic_name,
                :original_score,
                :practice_score,
                :improvement,
                :status,
                :recommendation
            )
        """),
        {
            "student_id": improvement_result["student_id"],
            "original_attempt_id": improvement_result[
                "original_attempt_id"
            ],
            "topic_id": None,
            "topic_name": improvement_result["topic"],
            "original_score": improvement_result[
                "original_score"
            ],
            "practice_score": improvement_result[
                "practice_score"
            ],
            "improvement": improvement_result[
                "improvement"
            ],
            "status": improvement_result["status"],
            "recommendation": improvement_result[
                "recommendation"
            ]
        }
    )


print("Learning progress saved successfully.")

Learning progress saved successfully.
